In [1]:
!pip install ultralytics
!pip install opencv-python
!pip install pillow
!pip install Roboflow


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


# Libraries

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt # For displaying images and plots
import matplotlib.animation as animation
import os
import shutil
import yaml
import cv2 # Used for image processing (e.g., BGR to RGB conversion)
import csv
import numpy as np
import json
import random # For selecting random examples
import joblib # For saving and loading the trained classifier
import pandas as pd
import torch # Import torch to check for CUDA availability
import seaborn as sns
import time # For frame processing speed
from PIL import Image
from ultralytics import YOLO
from collections import defaultdict
from roboflow import Roboflow
from pathlib import Path
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report, accuracy_score, f1_score, precision_score, recall_score
from sklearn.multioutput import MultiOutputClassifier
from sklearn.preprocessing import StandardScaler
from tqdm import tqdm # For progress bars
from dotenv import load_dotenv
from IPython.display import HTML, display
%matplotlib inline


In [3]:
load_dotenv()
ROBOFLOW_API_KEY = os.getenv("API_KEY")
rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace("squat-form-capstone").project("squat-detection-h1dwo")
version = project.version(19)

loading Roboflow workspace...
loading Roboflow project...


# Instructions for use :
My trained pose and classification models have been provided already at runs/pose/pose19_experiment/weight/best.pt

If you would like to load a new dataset and retrain the model you will need to use a roboflow API key and run the notebook sequentially.

If you wish to test the trained model on an image simply change the image input path of the "Image Analyzer" or a video input path of the "Video Analyzer" code blocks. Analyzed images and videos will be saved to /runs/pose/analyzed_images and analyze_videos respectively. Images/videos must exist and accessed locally with this current setup. Enjoy!

# Part 1: Pose Estimation Training

# Roboflow Data Downloader

In [4]:
def download_and_organize_dataset():
    # Step 1: Download dataset using Roboflow API
    dataset = version.download("yolov8")

    # Step 2: Create necessary directories
    folders = [
        "train/images", "valid/images", "test/images",
        "train/labels", "valid/labels", "test/labels",
        "configs", "models"
    ]
    for folder in folders:
        os.makedirs(folder, exist_ok=True)

    # Step 3: Move image/label files for each split
    base_path = dataset.location  # e.g., 'squat-detection-4'
    splits = {'train': 'train', 'valid': 'valid', 'test': 'test'}  # Map Roboflow -> your structure

    for split_src, split_dst in splits.items():
        image_src = os.path.join(base_path, split_src, 'images')
        label_src = os.path.join(base_path, split_src, 'labels')

        image_dst = os.path.join(split_dst, 'images')
        label_dst = os.path.join(split_dst, 'labels')

        # Move image files
        if os.path.exists(image_src):
            for file in os.listdir(image_src):
                shutil.move(os.path.join(image_src, file), os.path.join(image_dst, file))

        # Move label files
        if os.path.exists(label_src):
            for file in os.listdir(label_src):
                shutil.move(os.path.join(label_src, file), os.path.join(label_dst, file))

    # Step 4: Move data.yaml to configs/
    data_yaml_src = os.path.join(base_path, "data.yaml")
    data_yaml_dst = "configs/data.yaml"
    if os.path.exists(data_yaml_src):
        shutil.move(data_yaml_src, data_yaml_dst)
        print(f"Moved data.yaml to {data_yaml_dst}")

    shutil.rmtree(base_path)  # Clean up the original dataset folder

    print("Dataset fully organized and files moved successfully!")

if __name__ == "__main__":
    download_and_organize_dataset()


Extracting Dataset Version Zip to Squat-Detection-19 in yolov8:: 100%|██████████| 3814/3814 [00:00<00:00, 17118.25it/s]


Moved data.yaml to configs/data.yaml
Dataset fully organized and files moved successfully!


## Training Pose Estimation

In [ ]:
def train_custom_pose():
# Load a pretrained YOLO11 pose model
    model = YOLO('yolo26n-pose.pt')

    # Load your custom dataset configuration
    with open('./configs/data.yaml', 'r') as file:
        data_config = yaml.safe_load(file)

    print("Starting optimized training with 19 keypoints...")

    # Determine the device(s) to use for training
    if torch.cuda.is_available():
        num_gpus = torch.cuda.device_count()
        if num_gpus > 1:
            training_device = list(range(num_gpus)) # Use all available GPUs
            print(f"CUDA GPUs are available ({num_gpus} detected). Training will use devices: {training_device}.")
        else:
            training_device = 'cuda'
            print("CUDA GPU is available (1 detected). Training will use the GPU.")
    else:
        training_device = 'cpu'
        print("CUDA GPU is not available. Training will use the CPU.")

    # Train the model
    results = model.train(
        data="configs/data.yaml",    # Ensure your data.yaml is in this path
        epochs=100,                  # Start with 100; you can stop early if results plateau
        imgsz=640,                   # Standard resolution for squat posture
        device=training_device,      # Use 0 for single GPU or [0, 1] for Kaggle's Dual T4
        optimizer='MuSGD',           # Leveraging the 2026 optimized optimizer
        batch=16,                    # Adjust based on memory (T4=16GB, P100=16GB)
        workers=8,                   # Kaggle CPUs are decent; 4 workers avoids bottlenecks
        cache=True,
        project="squat-project",
        name="yolo26_pose_v4",
        exist_ok=True,               # Overwrite if folder exists
        pretrained=True              # Start from YOLO weights for faster learning
    )
    
    print("Training completed!")
    print(f"Best model saved at: {results.save_dir}")

if __name__ == "__main__":
    train_custom_pose()

Starting optimized training with 19 keypoints...
CUDA GPU is not available. Training will use the CPU.
Ultralytics 8.4.14 🚀 Python-3.11.14 torch-2.10.0+cu128 CPU (AMD Ryzen 9 8945HS w/ Radeon 780M Graphics)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=./configs/data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n-pose.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, 

KeyboardInterrupt: 

## Configurations & Paths

In [ ]:
model_checkpoint_path = "./runs/pose/pose19_experiment/weights/best.pt"

## Test Pose Estimation

### Evaluate pose and Metrics

In [ ]:
def evaluate_custom_pose():
    # Path to your dataset configuration YAML file
    data_yaml_path = './configs/data.yaml'

    # Define the output directory for evaluation results
    EVAL_PROJECT = 'runs/pose'
    EVAL_NAME = 'pose19_experiment_evaluation' # Choose a clear name for your evaluation run

    # --- Pre-checks ---
    if not os.path.exists(model_checkpoint_path):
        print(f"Error: Model checkpoint not found at {model_checkpoint_path}")
        print("Please ensure you have trained your model and the path is correct.")
        return

    if not os.path.exists(data_yaml_path):
        print(f"Error: Dataset YAML not found at {data_yaml_path}")
        print("Please ensure your data is organized and data.yaml is in 'configs/'")
        return

    # --- Load Model ---
    try:
        model = YOLO(model_checkpoint_path)
        print(f"Loaded model from: {model_checkpoint_path}")
    except Exception as e:
        print(f"Error loading model: {e}")
        return

    # --- Load Dataset Config and Print Info ---
    with open(data_yaml_path, 'r') as file:
        data_config = yaml.safe_load(file)

    print(f"Evaluating model on dataset configured by: {data_yaml_path}")
    print(f"Dataset root path: {data_config.get('path', 'Not specified in YAML')}")
    if 'test' in data_config:
        print(f"Using test set: {data_config['test']}")
    elif 'val' in data_config:
        print(f"Using validation set: {data_config['val']} (No 'test' split found)")
    else:
        print("Warning: Neither 'test' nor 'val' split found in data.yaml. Evaluation might fail.")

    # --- Determine Device ---
    evaluation_device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f"Evaluation will use the {evaluation_device.upper()}.")

    # --- Evaluate the Model ---
    try:
        results = model.val(
            data=data_yaml_path,
            imgsz=640, # Use the same image size as training
            batch=16,  # Adjust batch size based on your GPU/CPU memory
            device=evaluation_device,
            verbose=True,
            project=EVAL_PROJECT,
            name=EVAL_NAME,
            save_json=True # Needed if you want to also programmatically extract F1 threshold
        )

        print("\n--- Evaluation Metrics ---")
        print("Object Detection (Bounding Box) Metrics:")
        print(f"  mAP50-95: {results.box.map:.4f}")
        print(f"  mAP50:    {results.box.map50:.4f}")
        print(f"  mAP75:    {results.box.map75:.4f}")

        print("\nPose Estimation (Keypoint) Metrics:")
        print(f"  kpt_mAP50-95 (mAP^pose 50-95): {results.pose.map:.4f}")
        print(f"  kpt_mAP50 (mAP^pose 50):    {results.pose.map50:.4f}")

        # The results will be saved to the specified project/name directory
        results_save_dir = Path(results.save_dir)
        print(f"Evaluation results saved to: {results_save_dir}")

        # --- Display the Pose F1 Curve Graph ---
        # Corrected filename here:
        f1_curve_path = results_save_dir / 'PoseF1_curve.png' # <--- CHANGED THIS LINE

        if f1_curve_path.exists():
            try:
                img = Image.open(f1_curve_path)
                plt.figure(figsize=(10, 6)) # Optional: Adjust figure size
                plt.imshow(img)
                plt.axis('off') # Hide axes for cleaner image display
                plt.title("Pose Estimation F1 Curve (Confidence vs F1-score)")
                plt.show() # Display the plot window
                print(f"Displayed Pose F1 curve from: {f1_curve_path}")
            except Exception as plot_e:
                print(f"Error displaying Pose F1 curve: {plot_e}")
                print(f"Please check if {f1_curve_path} is a valid image or if matplotlib/Pillow are correctly installed.")
        else:
            print(f"Warning: PoseF1_curve.png not found at {f1_curve_path}.")
            print("Ensure evaluation completed successfully and the file was generated.")


    except Exception as e:
        print(f"An error occurred during evaluation: {e}")
        print("Please ensure your dataset configuration in data.yaml is correct and accessible.")


if __name__ == "__main__":
    evaluate_custom_pose()

### Observations
- Mean Average Precision (mAP) with thresholds 0.50-0.95 scored 76.65%

- Ideal confidence level is at .723 for optimal F1 score

In [ ]:
CONF_THRESHOLD = 0.723 # Adjusted based on evaluation results

### Test Pose Estimation on Single Image

In [ ]:
TEST_IMAGES_BASE_PATH = "test/images"

PERSON_19_KPT_NAMES = [
    "nose", "left_eye", "right_eye", "left_ear", "right_ear",
    "left_shoulder", "right_shoulder", "left_elbow", "right_elbow", "left_wrist",
    "right_wrist", "left_hip", "right_hip", "left_knee", "right_knee",
    "left_ankle", "right_ankle", "left_foot", "right_foot"
]

PERSON_19_SKELETON_CONNECTIONS = [
    # Head and Shoulders
    [0, 1], [0, 2], [1, 3], [2, 4], # Nose to eyes, eyes to ears
    [5, 6], # Shoulders
    # Arms
    [5, 7], [7, 9], # Left arm
    [6, 8], [8, 10], # Right arm
    # Torso
    [5, 11], [6, 12], # Shoulder to hip
    [11, 12], # Hips
    # Legs
    [11, 13], [13, 15], # Left leg
    [12, 14], [14, 16], # Right leg
    # Feet
    [15, 17], # Left ankle to left foot
    [16, 18] # Right ankle to right foot
]

# --- Drawing Constants for Skeleton ---
KEYPOINT_COLOR = (0, 0, 255) # Red for keypoints (easily visible)
LINE_COLOR = (255, 0, 0) # Blue for skeleton lines (contrasting)
RADIUS = 5 # Radius for keypoint circles
THICKNESS = 2 # Thickness for lines and keypoint outlines

# --- Drawing Constants for Keypoint Labels ---
LABEL_FONT = cv2.FONT_HERSHEY_SIMPLEX
LABEL_FONT_SCALE = 0.5 # Adjust for readability
LABEL_FONT_THICKNESS = 1
LABEL_COLOR = (0, 255, 0) # Green for labels (contrasting)
LABEL_OFFSET_X = 10 # Offset label from keypoint
LABEL_OFFSET_Y = -5 # Offset label from keypoint

def get_kpt_coords(idx, kpts_array, confidence_threshold):
    """
    Safely gets keypoint coordinates if confident.
    Returns (x,y) if confident, None otherwise.
    """
    if kpts_array is None or idx >= kpts_array.shape[0]:
        return None
    if kpts_array[idx, 2] > confidence_threshold:
        return kpts_array[idx, :2].astype(int) # Ensure integer coordinates for drawing
    return None

def test_custom_model_on_random_image():
    try:
        # 1. Load the trained YOLOv8 pose model
        model = YOLO(model_checkpoint_path)
        print("Custom 19-keypoint model loaded successfully!")

        print("Model metadata (keypoint names and skeleton connections) are defined for manual plotting.")

        # 2. Get a list of all image files in the test directory
        if not os.path.exists(TEST_IMAGES_BASE_PATH):
            print(f"Error: Test image folder not found at {TEST_IMAGES_BASE_PATH}")
            print("Please ensure your Roboflow dataset is downloaded and organized correctly.")
            return

        image_files = [f for f in os.listdir(TEST_IMAGES_BASE_PATH) if f.lower().endswith(('.png', '.jpg', '.jpeg', '.gif', '.bmp', '.tiff'))]

        if not image_files:
            print(f"Error: No image files found in {TEST_IMAGES_BASE_PATH}. Please check the folder content.")
            return

        # 3. Select a random image file
        random_image_filename = random.choice(image_files)
        image_path = os.path.join(TEST_IMAGES_BASE_PATH, random_image_filename)

        print(f"\nTesting on random image: {image_path}")

        # Check if the image can be read by OpenCV
        img_bgr_original = cv2.imread(image_path)
        if img_bgr_original is None:
            print(f"Error: OpenCV could not read image from {image_path}. It might be corrupted or not a valid image file.")
            return
        print(f"Debug: Image read successfully by OpenCV. Shape: {img_bgr_original.shape}, Type: {img_bgr_original.dtype}")


        # 4. Run inference on the selected image
        # Using the original image array directly for inference
        results = model(img_bgr_original, verbose=False)

        # 5. Process and display results for each detected object (person)
        for result in results: # 'results' is a list, but for single image, it's usually one result object
            num_persons_detected = len(result.boxes)
            best_person_confidence = 0.0
            num_keypoints_best_person = 0

            keypoints_data_for_metrics = None # For displaying confidences in text
            keypoints_xyc_to_draw = None # For actual drawing

            # Create a copy of the original image to draw on
            image_to_display_bgr = img_bgr_original.copy()

            if num_persons_detected > 0:
                best_box_idx = -1
                max_box_conf = -1.0
                for b_idx, box in enumerate(result.boxes):
                    if box.conf.item() > max_box_conf:
                        max_box_conf = box.conf.item()
                        best_box_idx = b_idx

                best_person_confidence = max_box_conf

                if best_box_idx != -1 and result.keypoints is not None and len(result.keypoints.data) > best_box_idx:
                    keypoints_data_for_metrics = result.keypoints.data[best_box_idx].cpu().numpy()
                    keypoints_xyc_to_draw = keypoints_data_for_metrics # Use the same keypoints for drawing
                    num_keypoints_best_person = keypoints_data_for_metrics.shape[0]

                    # --- Drawing Bounding Box ---
                    box_coords = result.boxes.xyxy[best_box_idx].cpu().numpy().astype(int)
                    cv2.rectangle(image_to_display_bgr, (box_coords[0], box_coords[1]),
                                  (box_coords[2], box_coords[3]), (0, 255, 255), 2) # Yellow box for visibility

                    # === MANUAL KEYPOINT, SKELETON, AND LABEL DRAWING ===
                    if keypoints_xyc_to_draw is not None:
                        # Draw keypoints (circles) and labels
                        for kpt_idx, (x, y, conf) in enumerate(keypoints_xyc_to_draw):
                            # Only draw keypoints if their confidence is above DRAWING_CONF_THRESHOLD
                            if conf > CONF_THRESHOLD:
                                cv2.circle(image_to_display_bgr, (int(x), int(y)), RADIUS, KEYPOINT_COLOR, -1)

                                # Draw keypoint label
                                kpt_name = PERSON_19_KPT_NAMES[kpt_idx] # Get the name
                                text_pos = (int(x) + LABEL_OFFSET_X, int(y) + LABEL_OFFSET_Y)
                                cv2.putText(image_to_display_bgr, kpt_name, text_pos,
                                            LABEL_FONT, LABEL_FONT_SCALE, LABEL_COLOR, LABEL_FONT_THICKNESS, cv2.LINE_AA)

                        # Draw skeleton lines
                        for connection in PERSON_19_SKELETON_CONNECTIONS: # Use your custom defined connections
                            p1_idx, p2_idx = connection

                            # Get coordinates, but this time check against DRAWING_CONF_THRESHOLD
                            # to determine if the points are confident enough for drawing a line.
                            # IMPORTANT: get_kpt_coords uses the DRAWING_CONF_THRESHOLD from its parameter
                            p1_coords_draw = get_kpt_coords(p1_idx, keypoints_xyc_to_draw, CONF_THRESHOLD)
                            p2_coords_draw = get_kpt_coords(p2_idx, keypoints_xyc_to_draw, CONF_THRESHOLD)

                            if p1_coords_draw is not None and p2_coords_draw is not None:
                                cv2.line(image_to_display_bgr, (int(p1_coords_draw[0]), int(p1_coords_draw[1])),
                                         (int(p2_coords_draw[0]), int(p2_coords_draw[1])), LINE_COLOR, THICKNESS)
                    else:
                        print("Debug: No confident keypoints found for the best person to draw skeleton.")
                    # ============================================

            # --- Prepare text for metrics display ---
            metrics_text = f"Detection Details for:\n{random_image_filename}\n\n"
            metrics_text += f"Persons Detected: {num_persons_detected}\n"
            if num_persons_detected > 0:
                metrics_text += f"Best Person Conf: {best_person_confidence:.2f}\n"
                metrics_text += f"Keypoints Found (Best Person): {num_keypoints_best_person}\n"

                metrics_text += "\nRelevant Keypoint Confidences:\n"
                # Updated to iterate PERSON_19_KPT_NAMES to match actual drawing
                for kpt_idx, kpt_name in enumerate(PERSON_19_KPT_NAMES):
                    if kpt_idx < num_keypoints_best_person:
                        confidence = keypoints_data_for_metrics[kpt_idx, 2]
                        # Only show if confidence is above the DRAWING threshold, for consistency
                        if confidence > CONF_THRESHOLD:
                            metrics_text += f"  {kpt_name}: {confidence:.2f}\n"
            else:
                metrics_text += "No confident person detection.\n"

            # Convert BGR image to RGB format for correct display with Matplotlib
            annotated_image_rgb = cv2.cvtColor(image_to_display_bgr, cv2.COLOR_BGR2RGB)

            print(f"Debug: Shape of annotated_image_rgb (before imshow): {annotated_image_rgb.shape}, Type: {annotated_image_rgb.dtype}")

            # Create a figure with two subplots (1 row, 2 columns) for side-by-side display
            fig, axes = plt.subplots(1, 2, figsize=(15, 8))

            # Plot the image in the first subplot
            axes[0].imshow(annotated_image_rgb)
            axes[0].set_title(f"Image: {os.path.basename(random_image_filename)}")
            axes[0].axis('off')

            # Display metrics text in the second subplot
            axes[1].text(0.05, 0.95, metrics_text,
                         verticalalignment='top',
                         fontsize=10,
                         fontfamily='monospace',
                         bbox=dict(boxstyle="round,pad=0.5", fc="white", ec="gray", lw=1, alpha=0.9))
            axes[1].set_title("Detection Metrics")
            axes[1].axis('off')

            plt.tight_layout()
            plt.show()

    except FileNotFoundError:
        print("Error: Trained model or test image folder not found. Please double check paths.")
        print(f"Model expected at: {os.path.abspath(model_checkpoint_path)}")
        print(f"Test image folder expected at: {os.path.abspath(TEST_IMAGES_BASE_PATH)}")
    except Exception as e:
        print(f"An unexpected error occurred: {e}")
        import traceback
        traceback.print_exc()


if __name__ == "__main__":
    test_custom_model_on_random_image()

# Part 2: Squat Analyzer (after Pose Estimation)

## Extract Image tags from roboflow

In [ ]:
def extract_coco_tags_and_organize():
    TARGET_COCO_JSON_BASE_PATH = os.path.join("configs", "coco_annotations")

    # --- Step 1: Download COCO Keypoints Format ---
    coco_download_format = "coco"
    try:
        print(f"Attempting to download COCO format '{coco_download_format}' (version {version})...")
        coco_dataset = version.download(coco_download_format)
        coco_base_path = coco_dataset.location
        print(f"COCO data successfully downloaded to temporary path: {coco_base_path}")
    except Exception as e:
        print(f"Error downloading COCO format '{coco_download_format}': {e}")
        print("Could not download COCO data. Please verify the project name, version, and available export formats on Roboflow website.")
        return

    # --- Step 2: Organize COCO JSON files ---
    print("\n--- Organizing COCO JSON files ---")
    splits_map = {'train': 'train', 'valid': 'val', 'test': 'test'}

    for split_src in splits_map.keys():
        src_json_path = os.path.join(coco_base_path, split_src, "_annotations.coco.json")
        dst_split_dir = os.path.join(TARGET_COCO_JSON_BASE_PATH, split_src)
        dst_json_path = os.path.join(dst_split_dir, "_annotations.coco.json")

        os.makedirs(dst_split_dir, exist_ok=True) # Create target directory for this split

        if os.path.exists(src_json_path):
            shutil.move(src_json_path, dst_json_path)
            print(f"Moved {split_src} JSON to: {dst_json_path}")
        else:
            print(f"Warning: Source JSON not found for {split_src} at: {src_json_path}. Skipping move.")

    # --- Step 3: Clean up the redundant Roboflow download folder ---
    print("\n--- Cleaning up temporary download folder ---")
    if coco_dataset and os.path.exists(coco_base_path) and os.path.isdir(coco_base_path):
        try:
            shutil.rmtree(coco_base_path)
            print(f"Cleaned up original COCO dataset folder: {coco_base_path}")
        except Exception as e:
            print(f"ERROR: Could not clean up {coco_base_path}: {e}")
    else:
        print("No COCO dataset folder to clean up (perhaps download failed or was manually cleaned).")

    # --- Step 4: Extract Tags from the NOW ORGANIZED COCO JSONs ---
    print("\n--- Starting Tag Extraction Per Split from organized JSONs ---")

    split_image_tags_ground_truth = defaultdict(dict)
    all_possible_tags_found = set()

    for split_src in splits_map.keys():
        current_json_path = os.path.join(TARGET_COCO_JSON_BASE_PATH, split_src, "_annotations.coco.json")

        print(f"Processing organized JSON file for {split_src} split: {current_json_path}")
        if os.path.exists(current_json_path):
            try:
                with open(current_json_path, 'r') as f:
                    coco_data = json.load(f)
                print(f"Successfully loaded JSON for {split_src} split.")
            except json.JSONDecodeError as e:
                print(f"ERROR: Could not decode JSON from {current_json_path}: {e}")
                continue

            if 'images' in coco_data:
                print(f"Extracting tags from {len(coco_data['images'])} images in {split_src} split.")
                for i, image_info in enumerate(coco_data['images']):
                    image_filename = image_info.get('file_name')

                    if not image_filename:
                        print(f"WARNING: Image {i} in {split_src} has no 'file_name'. Skipping.")
                        continue

                    # The keypoint to extract 'user_tags' from the 'extra' field
                    if 'extra' in image_info and 'user_tags' in image_info['extra']:
                        current_image_tags = image_info['extra']['user_tags']
                        if current_image_tags:
                            if i < 5: # Debug for first few images
                                print(f"  [{split_src}] Image '{image_filename}': Found user_tags: {current_image_tags}")

                            all_possible_tags_found.update(current_image_tags)

                            if image_filename not in split_image_tags_ground_truth[split_src]:
                                split_image_tags_ground_truth[split_src][image_filename] = []
                            split_image_tags_ground_truth[split_src][image_filename].extend(current_image_tags)
                            split_image_tags_ground_truth[split_src][image_filename] = list(set(split_image_tags_ground_truth[split_src][image_filename]))
                    else:
                        if i < 5: print(f"  [{split_src}] Image '{image_filename}': No 'extra' or 'user_tags' found.")
            else:
                print(f"No 'images' key found in {current_json_path} for {split_src}. Skipping tag extraction.")
        else:
            print(f"Organized JSON file not found for {split_src} split at: {current_json_path}")

    if not all_possible_tags_found:
        print("\n--- WARNING: No tags were extracted from any split. ---")
        print("Please verify your dataset's annotations and the COCO JSON structure.")
    else:
        print(f"\nSuccessfully extracted unique tags: {sorted(list(all_possible_tags_found))}")
        print("Collected tags per split.")


    # --- Step 5: Save Extracted Tags to CSV for each split ---
    print("\n--- Saving Tags to CSVs Per Split alongside JSONs ---")

    csv_fieldnames_base = ['image_filename']
    sorted_all_possible_tags = sorted(list(all_possible_tags_found))
    csv_fieldnames = csv_fieldnames_base + sorted_all_possible_tags

    for split_src, image_tags_data in split_image_tags_ground_truth.items():
        if image_tags_data:
            output_folder_path = os.path.join(TARGET_COCO_JSON_BASE_PATH, split_src)
            output_csv_path = os.path.join(output_folder_path, "image_tags_ground_truth.csv")

            # Directory already created in Step 2
            with open(output_csv_path, 'w', newline='') as csvfile:
                writer = csv.DictWriter(csvfile, fieldnames=csv_fieldnames)
                writer.writeheader()
                for filename, tags_list_for_image in image_tags_data.items():
                    row = {'image_filename': filename}
                    for tag in sorted_all_possible_tags:
                        row[tag] = 1 if tag in tags_list_for_image else 0
                    writer.writerow(row)

            print(f"Tags for '{split_src}' split saved to: {output_csv_path}")
        else:
            print(f"No tags found for '{split_src}' split. Skipping CSV creation for this split.")

    print("\nCOCO tag extraction process finished.")

if __name__ == "__main__":
    extract_coco_tags_and_organize()

### Distribution of Tags

In [ ]:
train_df = pd.read_csv('./configs/coco_annotations/train/image_tags_ground_truth.csv')
valid_df = pd.read_csv('./configs/coco_annotations/valid/image_tags_ground_truth.csv')
test_df = pd.read_csv('./configs/coco_annotations/test/image_tags_ground_truth.csv')

#### Training Data

In [ ]:
train_tag_counts = train_df.iloc[:, 1:].sum().sort_values(ascending=False)
plt.figure(figsize=(12, 6))
train_tag_counts.plot(kind='bar', color='skyblue')
plt.title('Tag Distribution in Train Set')
plt.xlabel('Tags')
plt.ylabel('Count')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

train_tag_counts = train_df.iloc[:, 1:].sum().sort_values(ascending=False)
for tag, count in train_tag_counts.items():
    print(f"Train Tag: {tag}, Count: {count}")

#### Validation Data

In [ ]:
valid_tag_counts = valid_df.iloc[:, 1:].sum().sort_values(ascending=False)
plt.figure(figsize=(12, 6))
valid_tag_counts.plot(kind='bar', color='lightgreen')
plt.title('Tag Distribution in Validation Set')
plt.xlabel('Tags')
plt.ylabel('Count')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

valid_tag_counts = valid_df.iloc[:, 1:].sum().sort_values(ascending=False)
for tag, count in valid_tag_counts.items():
    print(f"Validation Tag: {tag}, Count: {count}")

In [ ]:
train_tag_counts = train_df.iloc[:, 1:].sum().sort_values(ascending=False)
valid_tag_counts = valid_df.iloc[:, 1:].sum().sort_values(ascending=False)
tags = train_tag_counts.index

x = np.arange(len(tags))
width = 0.35
plt.figure(figsize=(12, 6))

# Plot Train vs Validation
plt.bar(x - width/2, train_tag_counts, width, label='Train', color='skyblue')
plt.bar(x + width/2, valid_tag_counts, width, label='Validation', color='lightgreen')
plt.title('Train vs Validation Tag Distribution')
plt.xlabel('Tags')
plt.ylabel('Count')
plt.xticks(x, tags, rotation=45)
plt.legend()
plt.tight_layout()
plt.show()

#### Testing Data

In [ ]:
test_tag_counts = test_df.iloc[:, 1:].sum().sort_values(ascending=False)
plt.figure(figsize=(12, 6))
test_tag_counts.plot(kind='bar', color='salmon')
plt.title('Tag Distribution in Test Set')
plt.xlabel('Tags')
plt.ylabel('Count')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

test_tag_counts = test_df.iloc[:, 1:].sum().sort_values(ascending=False)
for tag, count in test_tag_counts.items():
    print(f"Test Tag: {tag}, Count: {count}")

#### Summary of Dataset

In [ ]:
train_tag_counts = train_df.iloc[:, 1:].sum().sort_index()
valid_tag_counts = valid_df.iloc[:, 1:].sum().sort_index()
test_tag_counts = test_df.iloc[:, 1:].sum().sort_index()

tags = train_tag_counts.index


x = np.arange(len(tags))
width = 0.35

fig, axes = plt.subplots(1, 2, figsize=(20, 6), sharey=True)

# Plot Train vs Validation
axes[0].bar(x - width/2, train_tag_counts, width, label='Train', color='skyblue')
axes[0].bar(x + width/2, valid_tag_counts, width, label='Validation', color='lightgreen')
axes[0].set_title('Train vs Validation Tag Distribution')
axes[0].set_xlabel('Tags')
axes[0].set_ylabel('Count')
axes[0].set_xticks(x)
axes[0].set_xticklabels(tags, rotation=45)
axes[0].legend()

# Plot Test tag distribution
axes[1].bar(tags, test_tag_counts, color='salmon')
axes[1].set_title('Test Tag Distribution')
axes[1].set_xlabel('Tags')
axes[1].tick_params(axis='x', rotation=45)
axes[1].legend(['Test'])

# Adjust layout
plt.tight_layout()
plt.show()

df = pd.concat([train_df, valid_df, test_df], ignore_index=True)
Classifications = df.drop('image_filename', axis=1).sum().sort_values(ascending=False)
print(Classifications)

## Feature Engineering

In [ ]:
GROUND_TRUTH_BASE_PATH = os.path.join("configs", "coco_annotations")
NUM_FEATURES_EXPECTED = 19

def get_kpt_coords(idx, kpts_array, confidence_threshold):
    """Safely gets keypoint coordinates if confident."""
    if kpts_array is None or idx >= kpts_array.shape[0]:
        return None
    if kpts_array[idx, 2] > confidence_threshold:
        return kpts_array[idx, :2]
    return None

# --- Feature Engineering Function ---
def calculate_angle(p1, p2, p3):
    """Calculates angle between three points (P1-P2-P3)."""
    if p1 is None or p2 is None or p3 is None: return 0.0
    v1 = p1 - p2
    v2 = p3 - p2
    dot_product = np.dot(v1, v2)
    magnitude_v1 = np.linalg.norm(v1)
    magnitude_v2 = np.linalg.norm(v2)
    if magnitude_v1 == 0 or magnitude_v2 == 0: return 0.0
    cosine_angle = dot_product / (magnitude_v1 * magnitude_v2)
    angle_rad = np.arccos(np.clip(cosine_angle, -1.0, 1.0))
    return np.degrees(angle_rad)

def extract_keypoint_features(keypoints_xyc, img_width, img_height):
    """Extracts numerical features from YOLOv8 keypoint predictions."""
    if keypoints_xyc is None or keypoints_xyc.shape[0] < 19:
        return np.zeros(NUM_FEATURES_EXPECTED)

    # Define keypoint indices based on Person-19 schema
    NOSE, L_EYE, R_EYE, L_EAR, R_EAR = 0, 1, 2, 3, 4
    L_SHOULDER, R_SHOULDER, L_ELBOW, R_ELBOW, L_WRIST, R_WRIST = 5, 6, 7, 8, 9, 10
    L_HIP, R_HIP, L_KNEE, R_KNEE, L_ANKLE, R_ANKLE = 11, 12, 13, 14, 15, 16
    L_FOOT, R_FOOT = 17, 18

    kpts_dict = {
        'nose': get_kpt_coords(NOSE, keypoints_xyc, CONF_THRESHOLD),
        'l_shoulder': get_kpt_coords(L_SHOULDER, keypoints_xyc, CONF_THRESHOLD), 'r_shoulder': get_kpt_coords(R_SHOULDER, keypoints_xyc, CONF_THRESHOLD),
        'l_elbow': get_kpt_coords(L_ELBOW, keypoints_xyc, CONF_THRESHOLD), 'r_elbow': get_kpt_coords(R_ELBOW, keypoints_xyc, CONF_THRESHOLD),
        'l_wrist': get_kpt_coords(L_WRIST, keypoints_xyc, CONF_THRESHOLD), 'r_wrist': get_kpt_coords(R_WRIST, keypoints_xyc, CONF_THRESHOLD),
        'l_hip': get_kpt_coords(L_HIP, keypoints_xyc, CONF_THRESHOLD), 'r_hip': get_kpt_coords(R_HIP, keypoints_xyc, CONF_THRESHOLD),
        'l_knee': get_kpt_coords(L_KNEE, keypoints_xyc, CONF_THRESHOLD), 'r_knee': get_kpt_coords(R_KNEE, keypoints_xyc, CONF_THRESHOLD),
        'l_ankle': get_kpt_coords(L_ANKLE, keypoints_xyc, CONF_THRESHOLD), 'r_ankle': get_kpt_coords(R_ANKLE, keypoints_xyc, CONF_THRESHOLD),
        'l_foot': get_kpt_coords(L_FOOT, keypoints_xyc, CONF_THRESHOLD), 'r_foot': get_kpt_coords(R_FOOT, keypoints_xyc, CONF_THRESHOLD)
    }

    features = []
    # Lower Body Angles (Knees, Hips, Ankles)
    features.append(calculate_angle(kpts_dict['l_hip'], kpts_dict['l_knee'], kpts_dict['l_ankle']))
    features.append(calculate_angle(kpts_dict['r_hip'], kpts_dict['r_knee'], kpts_dict['r_ankle']))
    features.append(calculate_angle(kpts_dict['l_shoulder'], kpts_dict['l_hip'], kpts_dict['l_knee']))
    features.append(calculate_angle(kpts_dict['r_shoulder'], kpts_dict['r_hip'], kpts_dict['r_knee']))
    features.append(calculate_angle(kpts_dict['l_knee'], kpts_dict['l_ankle'], kpts_dict['l_foot']))
    features.append(calculate_angle(kpts_dict['r_knee'], kpts_dict['r_ankle'], kpts_dict['r_foot']))

    # Trunk Angle (Absolute angle relative to vertical)
    mid_shoulder = None
    mid_hip = None
    if kpts_dict['l_shoulder'] is not None and kpts_dict['r_shoulder'] is not None:
        mid_shoulder = (kpts_dict['l_shoulder'] + kpts_dict['r_shoulder']) / 2
    elif kpts_dict['l_shoulder'] is not None: mid_shoulder = kpts_dict['l_shoulder']
    elif kpts_dict['r_shoulder'] is not None: mid_shoulder = kpts_dict['r_shoulder']

    if kpts_dict['l_hip'] is not None and kpts_dict['r_hip'] is not None:
        mid_hip = (kpts_dict['l_hip'] + kpts_dict['r_hip']) / 2
    elif kpts_dict['l_hip'] is not None: mid_hip = kpts_dict['l_hip']
    elif kpts_dict['r_hip'] is not None: mid_hip = kpts_dict['r_hip']

    if mid_shoulder is not None and mid_hip is not None:
        trunk_vector = mid_shoulder - mid_hip
        trunk_angle = np.degrees(np.arctan2(trunk_vector[0], trunk_vector[1]))
        features.append(trunk_angle)
    else:
        features.append(0.0)

    # Left Femur Coronal Angle Proxy
    if kpts_dict['l_hip'] is not None and kpts_dict['l_knee'] is not None:
        l_knee_vertical_ref = np.array([kpts_dict['l_knee'][0], kpts_dict['l_knee'][1] + 100])
        features.append(calculate_angle(kpts_dict['l_hip'], kpts_dict['l_knee'], l_knee_vertical_ref))
    else:
        features.append(0.0)

    # Right Femur Coronal Angle Proxy
    if kpts_dict['r_hip'] is not None and kpts_dict['r_knee'] is not None:
        r_knee_vertical_ref = np.array([kpts_dict['r_knee'][0], kpts_dict['r_knee'][1] + 100])
        features.append(calculate_angle(kpts_dict['r_hip'], kpts_dict['r_knee'], r_knee_vertical_ref))
    else:
        features.append(0.0)

    # --- Forward Trunk: Torso_Horizontal_Lean_Ratio ---
    if mid_shoulder is not None and mid_hip is not None:
        horizontal_offset = mid_shoulder[0] - mid_hip[0]
        vertical_height = abs(mid_hip[1] - mid_shoulder[1])

        if vertical_height > 10: # Avoid division by zero
            torso_lean_ratio = horizontal_offset / vertical_height
            features.append(torso_lean_ratio)
        else:
            features.append(0.0)
    else:
        features.append(0.0)

    # --- Forward Trunk: Hip_Ankle_Horizontal_Offset_Ratio ---
    mid_ankle = None
    if kpts_dict['l_ankle'] is not None and kpts_dict['r_ankle'] is not None:
        mid_ankle = (kpts_dict['l_ankle'] + kpts_dict['r_ankle']) / 2
    elif kpts_dict['l_ankle'] is not None:
        mid_ankle = kpts_dict['l_ankle']
    elif kpts_dict['r_ankle'] is not None:
        mid_ankle = kpts_dict['r_ankle']

    if mid_hip is not None and mid_ankle is not None:
        hip_ankle_horizontal_offset = mid_hip[0] - mid_ankle[0]
        vertical_lower_body_height = abs(mid_hip[1] - mid_ankle[1])

        if vertical_lower_body_height > 10:
            hip_ankle_offset_ratio = hip_ankle_horizontal_offset / vertical_lower_body_height
            features.append(hip_ankle_offset_ratio)
        else:
            features.append(0.0)
    else:
        features.append(0.0)

    # Ankle_Foot_Vertical_Distance_Ratio (for Heels-Up) ---
    if kpts_dict['l_ankle'] is not None and kpts_dict['r_ankle'] is not None and \
       kpts_dict['l_foot'] is not None and kpts_dict['r_foot'] is not None and \
       mid_hip is not None and mid_ankle is not None: # Re-use mid_hip/mid_ankle for normalization

        avg_ankle_y = (kpts_dict['l_ankle'][1] + kpts_dict['r_ankle'][1]) / 2
        avg_foot_y = (kpts_dict['l_foot'][1] + kpts_dict['r_foot'][1]) / 2

        vertical_ankle_foot_dist = avg_ankle_y - avg_foot_y # Ankle is usually above foot (smaller y)

        # Normalize by the vertical height of the lower body to make it scale-invariant
        vertical_lower_body_height_norm = abs(mid_hip[1] - mid_ankle[1]) # From hip to ankle

        if vertical_lower_body_height_norm > 10: # Avoid division by zero
            ankle_foot_ratio = vertical_ankle_foot_dist / vertical_lower_body_height_norm
            features.append(ankle_foot_ratio)
        else:
            features.append(0.0)
    else:
        features.append(0.0)

    # Foot_Stability_Horizontal_Spread_Ratio ---
    if kpts_dict['l_foot'] is not None and kpts_dict['r_foot'] is not None and \
       kpts_dict['l_hip'] is not None and kpts_dict['r_hip'] is not None:

        horizontal_foot_dist = abs(kpts_dict['l_foot'][0] - kpts_dict['r_foot'][0])
        horizontal_hip_dist = abs(kpts_dict['l_hip'][0] - kpts_dict['r_hip'][0])

        if horizontal_hip_dist > 10: # Avoid division by zero
            foot_spread_ratio = horizontal_foot_dist / horizontal_hip_dist
            features.append(foot_spread_ratio)
        else:
            features.append(0.0)
    else:
        features.append(0.0)

    # Left Foot Orientation Angle
    if kpts_dict['l_ankle'] is not None and kpts_dict['l_foot'] is not None:
        vertical_ref_l = np.array([kpts_dict['l_ankle'][0], kpts_dict['l_ankle'][1] + 100])
        features.append(calculate_angle(vertical_ref_l, kpts_dict['l_ankle'], kpts_dict['l_foot']))
    else:
        features.append(0.0)

    # Right Foot Orientation Angle
    if kpts_dict['r_ankle'] is not None and kpts_dict['r_foot'] is not None:
        vertical_ref_r = np.array([kpts_dict['r_ankle'][0], kpts_dict['r_ankle'][1] + 100])
        features.append(calculate_angle(vertical_ref_r, kpts_dict['r_ankle'], kpts_dict['r_foot']))
    else:
        features.append(0.0)

    # Hip_Knee_Horizontal_Alignment_Ratio
    mid_knee = None
    if kpts_dict['l_knee'] is not None and kpts_dict['r_knee'] is not None:
        mid_knee = (kpts_dict['l_knee'] + kpts_dict['r_knee']) / 2
    elif kpts_dict['l_knee'] is not None:
        mid_knee = kpts_dict['l_knee']
    elif kpts_dict['r_knee'] is not None:
        mid_knee = kpts_dict['r_knee']

    if mid_hip is not None and mid_knee is not None:
        hip_knee_horizontal_offset = mid_hip[0] - mid_knee[0]
        vertical_hip_knee_distance = abs(mid_hip[1] - mid_knee[1])

        if vertical_hip_knee_distance > 10:
            hip_knee_alignment_ratio = hip_knee_horizontal_offset / vertical_hip_knee_distance
            features.append(hip_knee_alignment_ratio)
        else:
            features.append(0.0)
    else:
        features.append(0.0)

    # Left_Foot_Inclination_Angle (for Heels-Up) ---
    if kpts_dict['l_ankle'] is not None and kpts_dict['l_foot'] is not None:
        horizontal_ref_l = np.array([kpts_dict['l_ankle'][0] + 100, kpts_dict['l_ankle'][1]])
        features.append(calculate_angle(horizontal_ref_l, kpts_dict['l_ankle'], kpts_dict['l_foot']))
    else:
        features.append(0.0)

    # Right_Foot_Inclination_Angle (for Heels-Up) ---
    if kpts_dict['r_ankle'] is not None and kpts_dict['r_foot'] is not None:
        horizontal_ref_r = np.array([kpts_dict['r_ankle'][0] + 100, kpts_dict['r_ankle'][1]])
        features.append(calculate_angle(horizontal_ref_r, kpts_dict['r_ankle'], kpts_dict['r_foot']))
    else:
        features.append(0.0)

    # Knee_Ankle_Horizontal_Offset_Ratio (for Knee Travel) ---
    # Measures how far horizontally the knees are from the ankles, normalized by shin length.
    if mid_knee is not None and mid_ankle is not None:
        knee_ankle_horizontal_offset = mid_knee[0] - mid_ankle[0] # X-coordinate difference
        vertical_knee_ankle_distance = abs(mid_knee[1] - mid_ankle[1]) # Absolute Y-difference for length

        if vertical_knee_ankle_distance > 10: # Avoid division by zero or very small numbers
            knee_ankle_offset_ratio = knee_ankle_horizontal_offset / vertical_knee_ankle_distance
            features.append(knee_ankle_offset_ratio)
        else:
            features.append(0.0)
    else:
        features.append(0.0)


    # Pad or truncate features to ensure consistent length.
    while len(features) < NUM_FEATURES_EXPECTED:
        features.append(0.0)

    return np.array(features[:NUM_FEATURES_EXPECTED])

def load_features_and_labels_for_split(split_name, pose_model, all_unique_tags):
    """
    Loads image data and ground truth labels for a specified split,
    extracting pose features.
    """
    split_image_data = []
    split_ground_truth_labels = []
    split_image_filenames = []

    csv_path = os.path.join(GROUND_TRUTH_BASE_PATH, split_name, "image_tags_ground_truth.csv")
    image_folder_path = os.path.join(split_name, "images")

    if not os.path.exists(csv_path):
        print(f"WARNING: CSV file not found for {split_name} split at: {csv_path}. Skipping.")
        return split_image_data, split_ground_truth_labels, split_image_filenames

    if not os.path.exists(image_folder_path):
        print(f"WARNING: Image folder not found for {split_name} split at: {image_folder_path}. Skipping.")
        return split_image_data, split_ground_truth_labels, split_image_filenames

    try:
        df = pd.read_csv(csv_path)
        print(f"Loaded {len(df)} entries from {csv_path}")

        # Filter out 'image_filename' column only
        current_tags_in_csv = [col for col in df.columns if col != 'image_filename']

        # Ensure all unique tags (including those from other splits) are considered when building labels
        global_tag_mapping = {tag: i for i, tag in enumerate(sorted(list(all_unique_tags)))}

        for index, row in tqdm(df.iterrows(), total=len(df), desc=f"Processing {split_name} images for pose features"):
            image_filename = row['image_filename']
            full_image_path = os.path.join(image_folder_path, image_filename)

            if not os.path.exists(full_image_path):
                continue

            # Extract features
            results = pose_model(full_image_path, conf=CONF_THRESHOLD, verbose=False)
            features_for_image = None

            if results and len(results[0].keypoints) > 0:
                best_box_idx = -1
                max_box_conf = -1.0
                if results[0].boxes:
                    for b_idx, box in enumerate(results[0].boxes):
                        if box.conf.item() > max_box_conf:
                            max_box_conf = box.conf.item()
                            best_box_idx = b_idx

                if best_box_idx != -1:
                    keypoints_xyc_data = results[0].keypoints.data[best_box_idx].cpu().numpy()
                    img = cv2.imread(full_image_path)
                    if img is None:
                        continue
                    img_height, img_width, _ = img.shape
                    features_for_image = extract_keypoint_features(keypoints_xyc_data, img_width, img_height)

            if features_for_image is not None and len(features_for_image) == NUM_FEATURES_EXPECTED:
                # Create a binary label vector for this image based on ALL unique tags
                binary_label_vector = np.zeros(len(all_unique_tags))
                for tag_col in current_tags_in_csv:
                    if row[tag_col] == 1:
                        if tag_col in global_tag_mapping:
                            binary_label_vector[global_tag_mapping[tag_col]] = 1

                split_image_data.append(features_for_image)
                split_ground_truth_labels.append(binary_label_vector)
                split_image_filenames.append(image_filename)

    except pd.errors.EmptyDataError:
        print(f"WARNING: CSV file for {split_name} is empty: {csv_path}. Skipping.")
    except Exception as e:
        print(f"ERROR processing {split_name} split or its images/CSVs: {e}")

    return split_image_data, split_ground_truth_labels, split_image_filenames

def display_test_examples(pose_model, test_image_filenames, y_test, y_pred, class_names, image_base_paths_for_display, num_examples=5):
    print(f"\n--- Displaying {num_examples} Random Test Examples ---")
    if len(test_image_filenames) == 0:
        print("No test images available to display.")
        return
    SKELETON_CONNECTIONS = [
        (0, 1), (0, 2), (1, 3), (2, 4), (3, 5), (4, 6), (5, 7), (7, 9),
        (6, 8), (8, 10), (5, 11), (6, 12), (11, 13), (13, 15), (12, 14),
        (14, 16), (15, 17), (16, 18), (11, 12)
    ]
    KEYPOINT_COLOR = (0, 0, 255) # Blue
    LINE_COLOR = (255, 0, 0) # Red
    RADIUS = 5
    THICKNESS = 2

    num_examples = min(num_examples, len(test_image_filenames))
    if num_examples == 0:
        print("Not enough test images to display examples.")
        return

    random_indices = random.sample(range(len(test_image_filenames)), num_examples)
    plt.figure(figsize=(15, 6 * num_examples))

    for i, idx in enumerate(random_indices):
        img_filename = test_image_filenames[idx]
        true_labels_binary = y_test[idx]
        pred_labels_binary = y_pred[idx]

        true_tags = [class_names[j] for j, val in enumerate(true_labels_binary) if val == 1]
        pred_tags = [class_names[j] for j, val in enumerate(pred_labels_binary) if val == 1]

        image_path = os.path.join(image_base_paths_for_display['test'], img_filename)

        if not os.path.exists(image_path):
            continue

        try:
            img_bgr = cv2.imread(image_path)
            if img_bgr is None:
                continue

            results = pose_model(img_bgr, conf=CONF_THRESHOLD, verbose=False)

            keypoints_xyc_to_draw = None
            best_box_idx = -1
            max_box_conf = -1.0
            if results and results[0].boxes:
                for b_idx, box in enumerate(results[0].boxes):
                    if box.conf.item() > max_box_conf:
                        max_box_conf = box.conf.item()
                        best_box_idx = b_idx

            if best_box_idx != -1 and results[0].keypoints is not None and len(results[0].keypoints.data) > best_box_idx:
                keypoints_xyc_to_draw = results[0].keypoints.data[best_box_idx].cpu().numpy()

            display_conf_threshold = CONF_THRESHOLD

            if keypoints_xyc_to_draw is not None:
                for kpt_idx, (x, y, conf) in enumerate(keypoints_xyc_to_draw):
                    if conf > display_conf_threshold:
                        cv2.circle(img_bgr, (int(x), int(y)), RADIUS, KEYPOINT_COLOR, -1)
                for connection in SKELETON_CONNECTIONS:
                    p1_idx, p2_idx = connection
                    p1_coords = get_kpt_coords(p1_idx, keypoints_xyc_to_draw, display_conf_threshold)
                    p2_coords = get_kpt_coords(p2_idx, keypoints_xyc_to_draw, display_conf_threshold)
                    if p1_coords is not None and p2_coords is not None:
                        cv2.line(img_bgr, (int(p1_coords[0]), int(p1_coords[1])),
                                 (int(p2_coords[0]), int(p2_coords[1])), LINE_COLOR, THICKNESS)
            else:
                pass

            img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

            base_filename = img_filename.split('.rf.')[0] + os.path.splitext(img_filename)[1]
            title_text = f"Image: {base_filename}\nGT: {', '.join(true_tags) if true_tags else 'None'}\nPred: {', '.join(pred_tags) if pred_tags else 'None'}"

            plt.subplot(num_examples, 1, i + 1)
            plt.imshow(img_rgb)
            plt.title(title_text, fontsize=10)
            plt.axis('off')
        except Exception as e:
            print(f"Error displaying image {img_filename} or drawing keypoints: {e}")
            import traceback
            traceback.print_exc()
    plt.tight_layout()
    plt.show()

## Correlation Matrix

In [ ]:
def plot_feature_correlation_matrix(X_data, feature_names):
    """
    Generates and displays a correlation matrix heatmap for the given features.

    Args:
        X_data (np.array or pd.DataFrame): The feature data.
        feature_names (list): A list of strings, the names of the features,
                              in the same order as columns in X_data.
    """
    if X_data.shape[1] != len(feature_names):
        print("WARNING: Number of features in data does not match the number of provided feature names.")
        print("Expected:", len(feature_names), "Got:", X_data.shape[1])
        print("Correlation matrix labels might be incorrect.")

    df_features = pd.DataFrame(X_data, columns=feature_names)
    correlation_matrix = df_features.corr()

    plt.figure(figsize=(18, 16))
    sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt=".2f", linewidths=.5, annot_kws={"size": 8})

    plt.title('Feature Correlation Matrix', fontsize=20)
    plt.xticks(rotation=90, ha='right', fontsize=12)
    plt.yticks(rotation=0, va='center', fontsize=12)
    plt.tight_layout()
    plt.show()
    print("Feature correlation matrix displayed.")


if __name__ == "__main__":
    print("--- Starting Feature Correlation Matrix Generation with REAL DATA ---")

    # --- Load Pose Model ---
    try:
        pose_model = YOLO(model_checkpoint_path)
        print(f"Successfully loaded pose estimation model from: {model_checkpoint_path}")
    except Exception as e:
        print(f"ERROR: Could not load pose estimation model. Please check model_checkpoint_path: {e}")
        exit()

    # --- Collect ALL unique tags from all CSVs first (to ensure consistent label mapping) ---
    all_unique_tags = set()
    for split_name in ['train', 'valid', 'test']: # Check all splits to get a comprehensive list of tags
        csv_path = os.path.join(GROUND_TRUTH_BASE_PATH, split_name, "image_tags_ground_truth.csv")
        if os.path.exists(csv_path):
            try:
                df_temp = pd.read_csv(csv_path)
                current_tags_in_csv = [col for col in df_temp.columns if col != 'image_filename']
                all_unique_tags.update(current_tags_in_csv)
            except pd.errors.EmptyDataError:
                print(f"WARNING: CSV file for {split_name} is empty: {csv_path}. Skipping tag collection for this split.")
            except Exception as e:
                print(f"ERROR collecting tags from {split_name} CSV: {e}")

    if not all_unique_tags:
        print("ERROR: No unique tags found across any CSV files. Cannot proceed with feature loading.")
        exit()

    # --- Load Features from Training and Validation Data ---
    print("\n--- Loading Features from Train and Valid Splits ---")
    X_train_list, _, _ = [], [], []
    for split_name in ['train', 'valid']:
        x_s, _, _ = load_features_and_labels_for_split(split_name, pose_model, all_unique_tags)
        X_train_list.extend(x_s)

    X_data_for_plotting = np.array(X_train_list)
    print(f"Loaded {len(X_data_for_plotting)} samples for correlation analysis.")

    if X_data_for_plotting.shape[0] == 0:
        print("ERROR: No valid data samples loaded for correlation matrix. Cannot plot.")
        exit()

    # --- Feature Scaling ---
    scaler = StandardScaler()
    X_data_for_plotting_scaled = scaler.fit_transform(X_data_for_plotting)
    print("Features scaled using StandardScaler.")

    # --- Feature Names ---
    feature_names = [
        "Left_Knee_Angle", "Right_Knee_Angle",
        "Left_Hip_Angle", "Right_Hip_Angle",
        "Left_Ankle_Angle", "Right_Ankle_Angle",
        "Trunk_Angle",
        "Left_Femur_Q_Angle", "Right_Femur_Q_Angle",
        "Torso_Horizontal_Lean_Ratio", "Hip_Ankle_Horizontal_Offset_Ratio",
        "Ankle_Foot_Vertical_Distance_Ratio", "Foot_Stability_Horizontal_Spread_Ratio",
        "Left_Foot_Orientation_Angle", "Right_Foot_Orientation_Angle",
        "Hip_Knee_Horizontal_Alignment_Ratio",
        "Left_Foot_Inclination_Angle", "Right_Foot_Inclination_Angle",
        "Knee_Ankle_Horizontal_Offset_Ratio"
    ]

    plot_feature_correlation_matrix(X_data_for_plotting_scaled, feature_names)

    print("\n--- Correlation Matrix Generation Complete ---")


### Observations
- Obvious strong positive correlations between symmetrical sides relating to the same joint area, considering these are symmetrical squats
- Strong positive correlation between hip and knee angles (deeper the squat the "smaller the angles")
- Moderately strong negative correlation between trunk and hip angles
- Moderately strong positive correlation between Trunk angle and torso horizontal displacement
- Moderately strong positive correlation between hip-knee alignment and trunk angle
- Low correlation between Q angle and foot spread ratio
- Low correlation between ankle angle and trunk angle

## Squat Analyzer

## Train ML Squat Classifier


In [ ]:
MODELS_DIR = "models"
CLASSIFIER_FILENAME_PREFIX = "squat_classifier"
CLASS_NAMES_FILENAME_SUFFIX = "_class_names"
SCALER_FILENAME = "squat_classifier_scaler.joblib"
TRAINED_SCALER_SAVE_PATH = os.path.join(MODELS_DIR, SCALER_FILENAME)

def display_test_examples(pose_model, test_image_filenames, y_test, y_pred, class_names, image_base_paths_for_display, num_examples=5):
    print(f"\n--- Displaying {num_examples} Random Test Examples ---")
    if len(test_image_filenames) == 0:
        print("No test images available to display.")
        return
    SKELETON_CONNECTIONS = [
        (0, 1), (0, 2), (1, 3), (2, 4), (3, 5), (4, 6), (5, 7), (7, 9),
        (6, 8), (8, 10), (5, 11), (6, 12), (11, 13), (13, 15), (12, 14),
        (14, 16), (15, 17), (16, 18), (11, 12)
    ]
    KEYPOINT_COLOR = (0, 0, 255) # Blue
    LINE_COLOR = (255, 0, 0) # Red
    RADIUS = 5
    THICKNESS = 2

    num_examples = min(num_examples, len(test_image_filenames))
    if num_examples == 0:
        print("Not enough test images to display examples.")
        return

    random_indices = random.sample(range(len(test_image_filenames)), num_examples)
    plt.figure(figsize=(15, 6 * num_examples))

    for i, idx in enumerate(random_indices):
        img_filename = test_image_filenames[idx]
        true_labels_binary = y_test[idx]
        pred_labels_binary = y_pred[idx]

        true_tags = [class_names[j] for j, val in enumerate(true_labels_binary) if val == 1]
        pred_tags = [class_names[j] for j, val in enumerate(pred_labels_binary) if val == 1]

        image_path = os.path.join(image_base_paths_for_display['test'], img_filename)

        if not os.path.exists(image_path):
            continue

        try:
            img_bgr = cv2.imread(image_path)
            if img_bgr is None:
                continue

            results = pose_model(img_bgr, conf=CONF_THRESHOLD, verbose=False)

            keypoints_xyc_to_draw = None
            best_box_idx = -1
            max_box_conf = -1.0
            if results and results[0].boxes:
                for b_idx, box in enumerate(results[0].boxes):
                    if box.conf.item() > max_box_conf:
                        max_box_conf = box.conf.item()
                        best_box_idx = b_idx

            if best_box_idx != -1 and results[0].keypoints is not None and len(results[0].keypoints.data) > best_box_idx:
                keypoints_xyc_to_draw = results[0].keypoints.data[best_box_idx].cpu().numpy()

            display_conf_threshold = CONF_THRESHOLD

            if keypoints_xyc_to_draw is not None:
                for kpt_idx, (x, y, conf) in enumerate(keypoints_xyc_to_draw):
                    if conf > display_conf_threshold:
                        cv2.circle(img_bgr, (int(x), int(y)), RADIUS, KEYPOINT_COLOR, -1)
                for connection in SKELETON_CONNECTIONS:
                    p1_idx, p2_idx = connection
                    p1_coords = get_kpt_coords(p1_idx, keypoints_xyc_to_draw, display_conf_threshold)
                    p2_coords = get_kpt_coords(p2_idx, keypoints_xyc_to_draw, display_conf_threshold)
                    if p1_coords is not None and p2_coords is not None:
                        cv2.line(img_bgr, (int(p1_coords[0]), int(p1_coords[1])),
                                 (int(p2_coords[0]), int(p2_coords[1])), LINE_COLOR, THICKNESS)
            else:
                pass

            img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

            base_filename = img_filename.split('.rf.')[0] + os.path.splitext(img_filename)[1]
            title_text = f"Image: {base_filename}\nGT: {', '.join(true_tags) if true_tags else 'None'}\nPred: {', '.join(pred_tags) if pred_tags else 'None'}"

            plt.subplot(num_examples, 1, i + 1)
            plt.imshow(img_rgb)
            plt.title(title_text, fontsize=10)
            plt.axis('off')
        except Exception as e:
            print(f"Error displaying image {img_filename} or drawing keypoints: {e}")
            import traceback
            traceback.print_exc()
    plt.tight_layout()
    plt.show()

# --- Main Classification Script ---
def classify_squat_form(model_type='balanced_rfc'):
    print("--- Starting Squat Form Classification ---")
    try:
        pose_model = YOLO(model_checkpoint_path)
        print(f"Successfully loaded pose estimation model from: {model_checkpoint_path}")
    except Exception as e:
        print(f"ERROR: Could not load pose estimation model. Please check model_checkpoint_path: {e}")
        return

    # --- Step 1: Collect ALL unique tags from all CSVs first ---
    all_unique_tags = set()
    for split_name in ['train', 'valid', 'test']:
        csv_path = os.path.join(GROUND_TRUTH_BASE_PATH, split_name, "image_tags_ground_truth.csv")
        if os.path.exists(csv_path):
            try:
                df_temp = pd.read_csv(csv_path)
                current_tags_in_csv = [col for col in df_temp.columns if col != 'image_filename']
                all_unique_tags.update(current_tags_in_csv)
            except pd.errors.EmptyDataError:
                print(f"WARNING: CSV file for {split_name} is empty: {csv_path}. Skipping tag collection for this split.")
            except Exception as e:
                print(f"ERROR collecting tags from {split_name} CSV: {e}")

    if not all_unique_tags:
        print("ERROR: No unique tags found across any CSV files. Cannot proceed with classification.")
        return

    class_names = sorted(list(all_unique_tags))
    print(f"\nTotal unique tags identified across all data: {len(class_names)} -> {class_names}")

    # --- Step 2: Load Features and Labels for Train, Valid, and Test Splits ---
    print("\n--- Loading Features and Labels for Train, Valid, Test Splits ---")

    print("Loading data for training (from 'train' and 'valid' splits)...")
    X_train_list, y_train_list, train_filenames_list = [], [], []
    for split_name in ['train', 'valid']:
        x_s, y_s, f_s = load_features_and_labels_for_split(split_name, pose_model, all_unique_tags)
        X_train_list.extend(x_s)
        y_train_list.extend(y_s)
        train_filenames_list.extend(f_s)

    X_train = np.array(X_train_list)
    y_train = np.array(y_train_list)
    train_filenames = np.array(train_filenames_list)
    print(f"Loaded {len(X_train)} samples for training.")

    print("Loading data for testing (from 'test' split)...")
    x_test_list, y_test_list, test_filenames_list = load_features_and_labels_for_split('test', pose_model, all_unique_tags)

    X_test = np.array(x_test_list)
    y_test = np.array(y_test_list)
    test_filenames = np.array(test_filenames_list)
    print(f"Loaded {len(X_test)} samples for testing.")

    if X_train.shape[0] == 0:
        print("ERROR: No valid training samples. Cannot train classifier.")
        return
    if X_test.shape[0] == 0:
        print("ERROR: No valid test samples. Cannot evaluate classifier.")
        return

    print(f"\nTotal samples for training: {len(X_train)}")
    print(f"Total samples for testing: {len(X_test)}")

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    print("Features scaled using StandardScaler.")


    os.makedirs(MODELS_DIR, exist_ok=True)
    joblib.dump(scaler, TRAINED_SCALER_SAVE_PATH) # This now uses MODELS_DIR
    print(f"Scaler saved to: {os.path.abspath(TRAINED_SCALER_SAVE_PATH)}")

    # --- Step 3: Train Multi-label Classifier ---
    print(f"--- Training Multi-label Classifier ({model_type.replace('_', ' ').title()}) ---")

    base_classifier = None
    if model_type == 'logistic':
        base_classifier = LogisticRegression(solver='liblinear', random_state=42, max_iter=1000)
        classifier_name_for_save = "logistic"
    elif model_type == 'svc':
        base_classifier = SVC(probability=True, random_state=42) # probability=True needed for predict_proba if used
        classifier_name_for_save = "svc"
    elif model_type == 'nb': # Naive Bayes
        base_classifier = GaussianNB()
        classifier_name_for_save = "nb"
    elif model_type == 'knn': # K-Nearest Neighbors
        base_classifier = KNeighborsClassifier(n_neighbors=5) # Default n_neighbors=5, can be tuned
        classifier_name_for_save = "knn"
    elif model_type == 'rfc':
        base_classifier = RandomForestClassifier(
            random_state=42,
            n_estimators=100,
            max_depth=None,
            min_samples_split=5,
            min_samples_leaf=1,
        )
        classifier_name_for_save = "rfc"
    elif model_type == 'balanced_rfc':
        base_classifier = RandomForestClassifier(
            random_state=42,
            class_weight='balanced', # Handle class imbalance
            n_estimators=100,
            max_depth=None,
            min_samples_split=5,
            min_samples_leaf=1,
        )
        classifier_name_for_save = "balanced_rfc"
    else:
        print(f"ERROR: Unknown model_type: {model_type}. Please choose from 'logistic', 'svc', 'rfc', 'balanced_rfc', 'nb', 'knn'.")
        return

    current_classifier_save_path = os.path.join(MODELS_DIR, f"{CLASSIFIER_FILENAME_PREFIX}_{classifier_name_for_save}.joblib")
    current_class_names_path = os.path.join(MODELS_DIR, f"{CLASSIFIER_FILENAME_PREFIX}_{classifier_name_for_save}{CLASS_NAMES_FILENAME_SUFFIX}.joblib")

    classifier = MultiOutputClassifier(
        estimator=base_classifier,
        n_jobs=-1,
    )

    try:
        classifier.fit(X_train_scaled, y_train) # Train on scaled data
        print("Classifier training complete.")
        joblib.dump(classifier, current_classifier_save_path)
        joblib.dump(class_names, current_class_names_path)
        print(f"Trained classifier saved to: {os.path.abspath(current_classifier_save_path)}")
        print(f"Class names saved to: {os.path.abspath(current_class_names_path)}")

    except Exception as e:
        print(f"ERROR: Classifier training failed: {e}")
        import traceback
        traceback.print_exc()
        return

    # --- Step 4: Evaluate Classifier on Test Set ---
    y_pred = classifier.predict(X_test_scaled) # Predict using scaled test data

    print("\n--- Classification Evaluation ---")

    exact_accuracy = accuracy_score(y_test, y_pred)
    print(f"Exact Match Accuracy: {exact_accuracy:.4f}")
    f1_micro = f1_score(y_test, y_pred, average='micro', zero_division=0)
    f1_macro = f1_score(y_test, y_pred, average='macro', zero_division=0)
    f1_weighted = f1_score(y_test, y_pred, average='weighted', zero_division=0)
    precision_micro = precision_score(y_test, y_pred, average='micro', zero_division=0)
    recall_micro = recall_score(y_test, y_pred, average='micro', zero_division=0)
    print(f"F1-Score (Micro): {f1_micro:.4f}")
    print(f"F1-Score (Macro): {f1_macro:.4f}")
    print(f"F1-Score (Weighted): {f1_weighted:.4f}")
    print(f"Precision (Micro): {precision_micro:.4f}")
    print(f"Recall (Micro): {recall_micro:.4f}")
    print("Detailed Classification Report (per label):")
    print(classification_report(y_test, y_pred, target_names=class_names, zero_division=0))

    # --- Step 5: Display Test Examples ---
    image_base_paths_for_display = {
        'train': os.path.join('train', 'images'),
        'valid': os.path.join('valid', 'images'),
        'test': os.path.join('test', 'images')
    }
    display_test_examples(pose_model, test_filenames, y_test, y_pred, class_names, image_base_paths_for_display, num_examples=5)

    print("--- Squat Form Classification Finished ---")

if __name__ == "__main__":
    os.makedirs(MODELS_DIR, exist_ok=True)
    # classify_squat_form(model_type='logistic')
    # classify_squat_form(model_type='svc')
    # classify_squat_form(model_type='nb')
    # classify_squat_form(model_type='knn')
    # classify_squat_form(model_type='rfc')
    classify_squat_form(model_type='balanced_rfc')

### GridSearch for Balance RFC

In [ ]:
# # --- Helper Function for Keypoint Coordinate Retrieval ---
# def get_kpt_coords(idx, kpts_array, confidence_threshold):
#     """Safely gets keypoint coordinates if confident."""
#     if kpts_array is None or idx >= kpts_array.shape[0]:
#         return None
#     if kpts_array[idx, 2] > confidence_threshold:
#         return kpts_array[idx, :2]
#     return None

# # --- Feature Engineering Function ---
# def calculate_angle(p1, p2, p3):
#     """Calculates angle between three points (P1-P2-P3)."""
#     if p1 is None or p2 is None or p3 is None: return 0.0
#     v1 = p1 - p2
#     v2 = p3 - p2
#     dot_product = np.dot(v1, v2)
#     magnitude_v1 = np.linalg.norm(v1)
#     magnitude_v2 = np.linalg.norm(v2)
#     if magnitude_v1 == 0 or magnitude_v2 == 0: return 0.0
#     cosine_angle = dot_product / (magnitude_v1 * magnitude_v2)
#     angle_rad = np.arccos(np.clip(cosine_angle, -1.0, 1.0))
#     return np.degrees(angle_rad)

# def extract_keypoint_features(keypoints_xyc, img_width, img_height):
#     """Extracts numerical features from YOLOv8 keypoint predictions."""
#     if keypoints_xyc is None or keypoints_xyc.shape[0] < 19:
#         return np.zeros(NUM_FEATURES_EXPECTED)

#     # Define keypoint indices based on Person-19 schema
#     NOSE, L_EYE, R_EYE, L_EAR, R_EAR = 0, 1, 2, 3, 4
#     L_SHOULDER, R_SHOULDER, L_ELBOW, R_ELBOW, L_WRIST, R_WRIST = 5, 6, 7, 8, 9, 10
#     L_HIP, R_HIP, L_KNEE, R_KNEE, L_ANKLE, R_ANKLE = 11, 12, 13, 14, 15, 16
#     L_FOOT, R_FOOT = 17, 18

#     kpts_dict = {
#         'nose': get_kpt_coords(NOSE, keypoints_xyc, CONF_THRESHOLD),
#         'l_shoulder': get_kpt_coords(L_SHOULDER, keypoints_xyc, CONF_THRESHOLD), 'r_shoulder': get_kpt_coords(R_SHOULDER, keypoints_xyc, CONF_THRESHOLD),
#         'l_elbow': get_kpt_coords(L_ELBOW, keypoints_xyc, CONF_THRESHOLD), 'r_elbow': get_kpt_coords(R_ELBOW, keypoints_xyc, CONF_THRESHOLD),
#         'l_wrist': get_kpt_coords(L_WRIST, keypoints_xyc, CONF_THRESHOLD), 'r_wrist': get_kpt_coords(R_WRIST, keypoints_xyc, CONF_THRESHOLD),
#         'l_hip': get_kpt_coords(L_HIP, keypoints_xyc, CONF_THRESHOLD), 'r_hip': get_kpt_coords(R_HIP, keypoints_xyc, CONF_THRESHOLD),
#         'l_knee': get_kpt_coords(L_KNEE, keypoints_xyc, CONF_THRESHOLD), 'r_knee': get_kpt_coords(R_KNEE, keypoints_xyc, CONF_THRESHOLD),
#         'l_ankle': get_kpt_coords(L_ANKLE, keypoints_xyc, CONF_THRESHOLD), 'r_ankle': get_kpt_coords(R_ANKLE, keypoints_xyc, CONF_THRESHOLD),
#         'l_foot': get_kpt_coords(L_FOOT, keypoints_xyc, CONF_THRESHOLD), 'r_foot': get_kpt_coords(R_FOOT, keypoints_xyc, CONF_THRESHOLD)
#     }

#     features = []
#     # Lower Body Angles (Knees, Hips, Ankles)
#     features.append(calculate_angle(kpts_dict['l_hip'], kpts_dict['l_knee'], kpts_dict['l_ankle']))
#     features.append(calculate_angle(kpts_dict['r_hip'], kpts_dict['r_knee'], kpts_dict['r_ankle']))
#     features.append(calculate_angle(kpts_dict['l_shoulder'], kpts_dict['l_hip'], kpts_dict['l_knee']))
#     features.append(calculate_angle(kpts_dict['r_shoulder'], kpts_dict['r_hip'], kpts_dict['r_knee']))
#     features.append(calculate_angle(kpts_dict['l_knee'], kpts_dict['l_ankle'], kpts_dict['l_foot']))
#     features.append(calculate_angle(kpts_dict['r_knee'], kpts_dict['r_ankle'], kpts_dict['r_foot']))

#     # Trunk Angle (Absolute angle relative to vertical)
#     mid_shoulder = None
#     mid_hip = None
#     if kpts_dict['l_shoulder'] is not None and kpts_dict['r_shoulder'] is not None:
#         mid_shoulder = (kpts_dict['l_shoulder'] + kpts_dict['r_shoulder']) / 2
#     elif kpts_dict['l_shoulder'] is not None: mid_shoulder = kpts_dict['l_shoulder']
#     elif kpts_dict['r_shoulder'] is not None: mid_shoulder = kpts_dict['r_shoulder']

#     if kpts_dict['l_hip'] is not None and kpts_dict['r_hip'] is not None:
#         mid_hip = (kpts_dict['l_hip'] + kpts_dict['r_hip']) / 2
#     elif kpts_dict['l_hip'] is not None: mid_hip = kpts_dict['l_hip']
#     elif kpts_dict['r_hip'] is not None: mid_hip = kpts_dict['r_hip']

#     if mid_shoulder is not None and mid_hip is not None:
#         trunk_vector = mid_shoulder - mid_hip
#         trunk_angle = np.degrees(np.arctan2(trunk_vector[0], trunk_vector[1]))
#         features.append(trunk_angle)
#     else:
#         features.append(0.0)

#     # --- Genu Valgus Proxies (2D Coronal Angles) ---
#     # Left Femur Coronal Angle Proxy
#     if kpts_dict['l_hip'] is not None and kpts_dict['l_knee'] is not None:
#         l_knee_vertical_ref = np.array([kpts_dict['l_knee'][0], kpts_dict['l_knee'][1] + 100])
#         features.append(calculate_angle(kpts_dict['l_hip'], kpts_dict['l_knee'], l_knee_vertical_ref))
#     else:
#         features.append(0.0)

#     # Right Femur Coronal Angle Proxy
#     if kpts_dict['r_hip'] is not None and kpts_dict['r_knee'] is not None:
#         r_knee_vertical_ref = np.array([kpts_dict['r_knee'][0], kpts_dict['r_knee'][1] + 100])
#         features.append(calculate_angle(kpts_dict['r_hip'], kpts_dict['r_knee'], r_knee_vertical_ref))
#     else:
#         features.append(0.0)

#     # --- NEW FEATURE for Forward Trunk: Torso_Horizontal_Lean_Ratio ---
#     if mid_shoulder is not None and mid_hip is not None:
#         horizontal_offset = mid_shoulder[0] - mid_hip[0]
#         vertical_height = abs(mid_hip[1] - mid_shoulder[1])

#         if vertical_height > 10: # Avoid division by zero
#             torso_lean_ratio = horizontal_offset / vertical_height
#             features.append(torso_lean_ratio)
#         else:
#             features.append(0.0)
#     else:
#         features.append(0.0)

#         # --- NEW FEATURE: Hip_Knee_Horizontal_Alignment_Ratio (from previous turn) ---
#     mid_knee = None
#     if kpts_dict['l_knee'] is not None and kpts_dict['r_knee'] is not None:
#         mid_knee = (kpts_dict['l_knee'] + kpts_dict['r_knee']) / 2
#     elif kpts_dict['l_knee'] is not None:
#         mid_knee = kpts_dict['l_knee']
#     elif kpts_dict['r_knee'] is not None:
#         mid_knee = kpts_dict['r_knee']

#     if mid_hip is not None and mid_knee is not None:
#         hip_knee_horizontal_offset = mid_hip[0] - mid_knee[0]
#         vertical_hip_knee_distance = abs(mid_hip[1] - mid_knee[1])

#         if vertical_hip_knee_distance > 10:
#             hip_knee_alignment_ratio = hip_knee_horizontal_offset / vertical_hip_knee_distance
#             features.append(hip_knee_alignment_ratio)
#         else:
#             features.append(0.0)
#     else:
#         features.append(0.0)

#     # --- NEW FEATURE for Forward Trunk: Hip_Ankle_Horizontal_Offset_Ratio ---
#     mid_ankle = None
#     if kpts_dict['l_ankle'] is not None and kpts_dict['r_ankle'] is not None:
#         mid_ankle = (kpts_dict['l_ankle'] + kpts_dict['r_ankle']) / 2
#     elif kpts_dict['l_ankle'] is not None:
#         mid_ankle = kpts_dict['l_ankle']
#     elif kpts_dict['r_ankle'] is not None:
#         mid_ankle = kpts_dict['r_ankle']

#     if mid_hip is not None and mid_ankle is not None:
#         hip_ankle_horizontal_offset = mid_hip[0] - mid_ankle[0]
#         vertical_lower_body_height = abs(mid_hip[1] - mid_ankle[1])

#         if vertical_lower_body_height > 10:
#             hip_ankle_offset_ratio = hip_ankle_horizontal_offset / vertical_lower_body_height
#             features.append(hip_ankle_offset_ratio)
#         else:
#             features.append(0.0)
#     else:
#         features.append(0.0)

#     # --- NEW FEATURE for Feet Movement: Ankle_Foot_Vertical_Distance_Ratio (for Heels-Up) ---
#     if kpts_dict['l_ankle'] is not None and kpts_dict['r_ankle'] is not None and \
#        kpts_dict['l_foot'] is not None and kpts_dict['r_foot'] is not None and \
#        mid_hip is not None and mid_ankle is not None: # Re-use mid_hip/mid_ankle for normalization

#         avg_ankle_y = (kpts_dict['l_ankle'][1] + kpts_dict['r_ankle'][1]) / 2
#         avg_foot_y = (kpts_dict['l_foot'][1] + kpts_dict['r_foot'][1]) / 2

#         vertical_ankle_foot_dist = avg_ankle_y - avg_foot_y # Ankle is usually above foot (smaller y)

#         # Normalize by the vertical height of the lower body to make it scale-invariant
#         vertical_lower_body_height_norm = abs(mid_hip[1] - mid_ankle[1]) # From hip to ankle

#         if vertical_lower_body_height_norm > 10: # Avoid division by zero
#             ankle_foot_ratio = vertical_ankle_foot_dist / vertical_lower_body_height_norm
#             features.append(ankle_foot_ratio)
#         else:
#             features.append(0.0)
#     else:
#         features.append(0.0)

#       # --- NEW FEATURE: Left_Foot_Inclination_Angle (for Heels-Up) ---
#     # Angle of the line segment from ankle to foot (ball) relative to horizontal.
#     # A larger angle suggests the heel is lifting.
#     if kpts_dict['l_ankle'] is not None and kpts_dict['l_foot'] is not None:
#         # Create a horizontal reference point from the ankle
#         horizontal_ref_l = np.array([kpts_dict['l_ankle'][0] + 100, kpts_dict['l_ankle'][1]]) # 100 pixels to the right
#         # Calculate angle around the ankle point: (horizontal_ref)-(ankle)-(foot)
#         features.append(calculate_angle(horizontal_ref_l, kpts_dict['l_ankle'], kpts_dict['l_foot']))
#     else:
#         features.append(0.0)

#     # --- NEW FEATURE: Right_Foot_Inclination_Angle (for Heels-Up) ---
#     if kpts_dict['r_ankle'] is not None and kpts_dict['r_foot'] is not None:
#         horizontal_ref_r = np.array([kpts_dict['r_ankle'][0] + 100, kpts_dict['r_ankle'][1]]) # 100 pixels to the right
#         features.append(calculate_angle(horizontal_ref_r, kpts_dict['r_ankle'], kpts_dict['r_foot']))
#     else:
#         features.append(0.0)

#     # --- NEW FEATURE for Feet Movement: Foot_Stability_Horizontal_Spread_Ratio ---
#     if kpts_dict['l_foot'] is not None and kpts_dict['r_foot'] is not None and \
#        kpts_dict['l_hip'] is not None and kpts_dict['r_hip'] is not None:

#         horizontal_foot_dist = abs(kpts_dict['l_foot'][0] - kpts_dict['r_foot'][0])
#         horizontal_hip_dist = abs(kpts_dict['l_hip'][0] - kpts_dict['r_hip'][0])

#         if horizontal_hip_dist > 10: # Avoid division by zero
#             foot_spread_ratio = horizontal_foot_dist / horizontal_hip_dist
#             features.append(foot_spread_ratio)
#         else:
#             features.append(0.0)
#     else:
#         features.append(0.0)

#     # --- NEW FEATURES for Foot Orientation: Left_Foot_Orientation_Angle & Right_Foot_Orientation_Angle ---
#     # Left Foot Orientation Angle
#     if kpts_dict['l_ankle'] is not None and kpts_dict['l_foot'] is not None:
#         # Define a vertical reference vector pointing downwards from the ankle
#         vertical_ref_l = np.array([kpts_dict['l_ankle'][0], kpts_dict['l_ankle'][1] + 100]) # 100 pixels down
#         features.append(calculate_angle(vertical_ref_l, kpts_dict['l_ankle'], kpts_dict['l_foot']))
#     else:
#         features.append(0.0)

#     # Right Foot Orientation Angle
#     if kpts_dict['r_ankle'] is not None and kpts_dict['r_foot'] is not None:
#         # Define a vertical reference vector pointing downwards from the ankle
#         vertical_ref_r = np.array([kpts_dict['r_ankle'][0], kpts_dict['r_ankle'][1] + 100]) # 100 pixels down
#         features.append(calculate_angle(vertical_ref_r, kpts_dict['r_ankle'], kpts_dict['r_foot']))
#     else:
#         features.append(0.0)


#     # Pad or truncate features to ensure consistent length.
#     while len(features) < NUM_FEATURES_EXPECTED:
#         features.append(0.0)

#     return np.array(features[:NUM_FEATURES_EXPECTED])


# # --- Helper Function to Load Features and Labels for a Given Split ---
# def load_features_and_labels_for_split(split_name, pose_model, all_unique_tags):
#     """
#     Loads image data and ground truth labels for a specified split,
#     extracting pose features.
#     """
#     split_image_data = []
#     split_ground_truth_labels = []
#     split_image_filenames = []

#     csv_path = os.path.join(GROUND_TRUTH_BASE_PATH, split_name, "image_tags_ground_truth.csv")
#     image_folder_path = os.path.join(split_name, "images")

#     if not os.path.exists(csv_path):
#         print(f"WARNING: CSV file not found for {split_name} split at: {csv_path}. Skipping.")
#         return split_image_data, split_ground_truth_labels, split_image_filenames

#     if not os.path.exists(image_folder_path):
#         print(f"WARNING: Image folder not found for {split_name} split at: {image_folder_path}. Skipping.")
#         return split_image_data, split_ground_truth_labels, split_image_filenames

#     try:
#         df = pd.read_csv(csv_path)
#         print(f"Loaded {len(df)} entries from {csv_path}")

#         # Filter out 'image_filename' column only
#         current_tags_in_csv = [col for col in df.columns if col != 'image_filename']

#         # Ensure all unique tags (including those from other splits) are considered when building labels
#         global_tag_mapping = {tag: i for i, tag in enumerate(sorted(list(all_unique_tags)))}

#         for index, row in tqdm(df.iterrows(), total=len(df), desc=f"Processing {split_name} images for pose features"):
#             image_filename = row['image_filename']
#             full_image_path = os.path.join(image_folder_path, image_filename)

#             if not os.path.exists(full_image_path):
#                 continue

#             # Extract features
#             results = pose_model(full_image_path, conf=CONF_THRESHOLD, verbose=False)
#             features_for_image = None

#             if results and len(results[0].keypoints) > 0:
#                 best_box_idx = -1
#                 max_box_conf = -1.0
#                 if results[0].boxes:
#                     for b_idx, box in enumerate(results[0].boxes):
#                         if box.conf.item() > max_box_conf:
#                             max_box_conf = box.conf.item()
#                             best_box_idx = b_idx

#                 if best_box_idx != -1:
#                     keypoints_xyc_data = results[0].keypoints.data[best_box_idx].cpu().numpy()
#                     img = cv2.imread(full_image_path)
#                     if img is None:
#                         continue
#                     img_height, img_width, _ = img.shape
#                     features_for_image = extract_keypoint_features(keypoints_xyc_data, img_width, img_height)

#             if features_for_image is not None and len(features_for_image) == NUM_FEATURES_EXPECTED:
#                 # Create a binary label vector for this image based on ALL unique tags
#                 binary_label_vector = np.zeros(len(all_unique_tags))
#                 for tag_col in current_tags_in_csv:
#                     if row[tag_col] == 1:
#                         if tag_col in global_tag_mapping:
#                             binary_label_vector[global_tag_mapping[tag_col]] = 1

#                 split_image_data.append(features_for_image)
#                 split_ground_truth_labels.append(binary_label_vector)
#                 split_image_filenames.append(image_filename)

#     except pd.errors.EmptyDataError:
#         print(f"WARNING: CSV file for {split_name} is empty: {csv_path}. Skipping.")
#     except Exception as e:
#         print(f"ERROR processing {split_name} split or its images/CSVs: {e}")

#     return split_image_data, split_ground_truth_labels, split_image_filenames


# # --- Helper Function to Display Examples ---
# def display_test_examples(pose_model, test_image_filenames, y_test, y_pred, class_names, image_base_paths_for_display, num_examples=5):
#     print(f"\n--- Displaying {num_examples} Random Test Examples ---")
#     if len(test_image_filenames) == 0:
#         print("No test images available to display.")
#         return
#     SKELETON_CONNECTIONS = [
#         (0, 1), (0, 2), (1, 3), (2, 4), (3, 5), (4, 6), (5, 7), (7, 9),
#         (6, 8), (8, 10), (5, 11), (6, 12), (11, 13), (13, 15), (12, 14),
#         (14, 16), (15, 17), (16, 18), (11, 12)
#     ]
#     KEYPOINT_COLOR = (0, 0, 255) # Blue
#     LINE_COLOR = (255, 0, 0) # Red
#     RADIUS = 5
#     THICKNESS = 2

#     num_examples = min(num_examples, len(test_image_filenames))
#     if num_examples == 0:
#         print("Not enough test images to display examples.")
#         return

#     random_indices = random.sample(range(len(test_image_filenames)), num_examples)
#     plt.figure(figsize=(15, 6 * num_examples))

#     for i, idx in enumerate(random_indices):
#         img_filename = test_image_filenames[idx]
#         true_labels_binary = y_test[idx]
#         pred_labels_binary = y_pred[idx]

#         true_tags = [class_names[j] for j, val in enumerate(true_labels_binary) if val == 1]
#         pred_tags = [class_names[j] for j, val in enumerate(pred_labels_binary) if val == 1]

#         image_path = os.path.join(image_base_paths_for_display['test'], img_filename)

#         if not os.path.exists(image_path):
#             continue

#         try:
#             img_bgr = cv2.imread(image_path)
#             if img_bgr is None:
#                 continue

#             results = pose_model(img_bgr, conf=CONF_THRESHOLD, verbose=False)

#             keypoints_xyc_to_draw = None
#             best_box_idx = -1
#             max_box_conf = -1.0
#             if results and results[0].boxes:
#                 for b_idx, box in enumerate(results[0].boxes):
#                     if box.conf.item() > max_box_conf:
#                         max_box_conf = box.conf.item()
#                         best_box_idx = b_idx

#             if best_box_idx != -1 and results[0].keypoints is not None and len(results[0].keypoints.data) > best_box_idx:
#                 keypoints_xyc_to_draw = results[0].keypoints.data[best_box_idx].cpu().numpy()

#             display_conf_threshold = CONF_THRESHOLD

#             if keypoints_xyc_to_draw is not None:
#                 for kpt_idx, (x, y, conf) in enumerate(keypoints_xyc_to_draw):
#                     if conf > display_conf_threshold:
#                         cv2.circle(img_bgr, (int(x), int(y)), RADIUS, KEYPOINT_COLOR, -1)
#                 for connection in SKELETON_CONNECTIONS:
#                     p1_idx, p2_idx = connection
#                     p1_coords = get_kpt_coords(p1_idx, keypoints_xyc_to_draw, display_conf_threshold)
#                     p2_coords = get_kpt_coords(p2_idx, keypoints_xyc_to_draw, display_conf_threshold)
#                     if p1_coords is not None and p2_coords is not None:
#                         cv2.line(img_bgr, (int(p1_coords[0]), int(p1_coords[1])),
#                                  (int(p2_coords[0]), int(p2_coords[1])), LINE_COLOR, THICKNESS)
#             else:
#                 pass

#             img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

#             base_filename = img_filename.split('.rf.')[0] + os.path.splitext(img_filename)[1]
#             title_text = f"Image: {base_filename}\nGT: {', '.join(true_tags) if true_tags else 'None'}\nPred: {', '.join(pred_tags) if pred_tags else 'None'}"

#             plt.subplot(num_examples, 1, i + 1)
#             plt.imshow(img_rgb)
#             plt.title(title_text, fontsize=10)
#             plt.axis('off')
#         except Exception as e:
#             print(f"Error displaying image {img_filename} or drawing keypoints: {e}")
#             import traceback
#             traceback.print_exc()
#     plt.tight_layout()
#     plt.show()


# # --- Main Classification Script ---
# def classify_squat_form():
#     print("--- Starting Squat Form Classification (Balanced RFC Tuned) ---")
#     try:
#         pose_model = YOLO(POSE_MODEL_PATH)
#         print(f"Successfully loaded pose estimation model from: {POSE_MODEL_PATH}")
#     except Exception as e:
#         print(f"ERROR: Could not load pose estimation model. Please check POSE_MODEL_PATH: {e}")
#         return

#     # --- Step 1: Collect ALL unique tags from all CSVs first ---
#     all_unique_tags = set()
#     for split_name in ['train', 'valid', 'test']:
#         csv_path = os.path.join(GROUND_TRUTH_BASE_PATH, split_name, "image_tags_ground_truth.csv")
#         if os.path.exists(csv_path):
#             try:
#                 df_temp = pd.read_csv(csv_path)
#                 current_tags_in_csv = [col for col in df_temp.columns if col != 'image_filename']
#                 all_unique_tags.update(current_tags_in_csv)
#             except pd.errors.EmptyDataError:
#                 print(f"WARNING: CSV file for {split_name} is empty: {csv_path}. Skipping tag collection for this split.")
#             except Exception as e:
#                 print(f"ERROR collecting tags from {split_name} CSV: {e}")

#     if not all_unique_tags:
#         print("ERROR: No unique tags found across any CSV files. Cannot proceed with classification.")
#         return

#     class_names = sorted(list(all_unique_tags))
#     print(f"\nTotal unique tags identified across all data: {len(class_names)} -> {class_names}")

#     # --- Step 2: Load Features and Labels for Train, Valid, and Test Splits ---
#     print("\n--- Loading Features and Labels for Train, Valid, Test Splits ---")

#     print("Loading data for training (from 'train' and 'valid' splits)...")
#     X_train_list, y_train_list, train_filenames_list = [], [], []
#     for split_name in ['train', 'valid']:
#         x_s, y_s, f_s = load_features_and_labels_for_split(split_name, pose_model, all_unique_tags)
#         X_train_list.extend(x_s)
#         y_train_list.extend(y_s)
#         train_filenames_list.extend(f_s)

#     X_train = np.array(X_train_list)
#     y_train = np.array(y_train_list)
#     train_filenames = np.array(train_filenames_list)
#     print(f"Loaded {len(X_train)} samples for training.")

#     print("Loading data for testing (from 'test' split)...")
#     x_test_list, y_test_list, test_filenames_list = load_features_and_labels_for_split('test', pose_model, all_unique_tags)

#     X_test = np.array(x_test_list)
#     y_test = np.array(y_test_list)
#     test_filenames = np.array(test_filenames_list)
#     print(f"Loaded {len(X_test)} samples for testing.")

#     if X_train.shape[0] == 0:
#         print("ERROR: No valid training samples. Cannot train classifier.")
#         return
#     if X_test.shape[0] == 0:
#         print("ERROR: No valid test samples. Cannot evaluate classifier.")
#         return

#     print(f"\nTotal samples for training: {len(X_train)}")
#     print(f"Total samples for testing: {len(X_test)}")

#     # --- Feature Scaling (Added StandardScaler) ---
#     scaler = StandardScaler()
#     X_train_scaled = scaler.fit_transform(X_train)
#     X_test_scaled = scaler.transform(X_test)
#     print("Features scaled using StandardScaler.")

#     os.makedirs(os.path.dirname(TRAINED_SCALER_SAVE_PATH), exist_ok=True)
#     joblib.dump(scaler, TRAINED_SCALER_SAVE_PATH)
#     print(f"Scaler saved to: {os.path.abspath(TRAINED_SCALER_SAVE_PATH)}")

#     # --- Step 3: Train Multi-label Classifier (Balanced Random Forest with GridSearchCV) ---
#     print(f"\n--- Performing GridSearchCV for Balanced Random Forest Classifier ---")
#     base_rfc_for_grid = RandomForestClassifier(random_state=42, class_weight='balanced')

#     # Define the parameter grid for GridSearchCV
#     param_grid = {
#         'estimator__n_estimators': [50, 100, 200], # Number of trees
#         'estimator__max_depth': [10, 20, None],   # Max depth of trees
#         'estimator__min_samples_split': [2, 5],   # Min samples to split a node
#         'estimator__min_samples_leaf': [1, 2],    # Min samples at a leaf node
#         # 'estimator__max_features': ['sqrt', 'log2', 0.8], # Uncomment to tune this too
#     }

#     # MultiOutputClassifier needs to wrap the base estimator for GridSearchCV
#     multi_output_estimator = MultiOutputClassifier(estimator=base_rfc_for_grid, n_jobs=-1)

#     # Using F1-macro as scoring metric, crucial for multi-label and imbalance
#     grid_search = GridSearchCV(
#         estimator=multi_output_estimator,
#         param_grid=param_grid,
#         scoring='f1_macro', # Important for multi-label, imbalanced datasets
#         cv=3,               # 3-fold cross-validation (adjust based on data size/computation)
#         verbose=2,          # Print progress
#         n_jobs=-1           # Use all available CPU cores
#     )

#     try:
#         grid_search.fit(X_train_scaled, y_train)
#         print("\nGridSearchCV complete.")
#         print(f"Best hyperparameters found: {grid_search.best_params_}")
#         print(f"Best cross-validation F1-Macro score: {grid_search.best_score_:.4f}")
#         classifier = grid_search.best_estimator_ # Get the best model
#         classifier_name_for_save = "balanced_rfc_tuned"

#     except Exception as e:
#         print(f"ERROR: GridSearchCV failed: {e}")
#         import traceback
#         traceback.print_exc()
#         return

#     # Skip saving if GridSearchCV failed and returned None
#     if classifier is None:
#         return

#     current_classifier_save_path = os.path.join(os.path.dirname(TRAINED_CLASSIFIER_SAVE_BASE_PATH), f"{TRAINED_CLASSIFIER_SAVE_BASE_PATH.split('/')[-1]}_{classifier_name_for_save}.joblib")
#     current_class_names_path = os.path.join(os.path.dirname(TRAINED_CLASS_NAMES_BASE_PATH), f"{TRAINED_CLASS_NAMES_BASE_PATH.split('/')[-1]}_{classifier_name_for_save}_class_names.joblib")


#     try:
#         os.makedirs(os.path.dirname(current_classifier_save_path), exist_ok=True)

#         joblib.dump(classifier, current_classifier_save_path)
#         joblib.dump(class_names, current_class_names_path)
#         print(f"Trained classifier saved to: {os.path.abspath(current_classifier_save_path)}")
#         print(f"Class names saved to: {os.path.abspath(current_class_names_path)}")

#     except Exception as e:
#         print(f"ERROR: Classifier training/saving failed: {e}")
#         import traceback
#         traceback.print_exc()
#         return

#     # --- Step 4: Evaluate Classifier on Test Set ---
#     y_pred = classifier.predict(X_test_scaled) # Predict using scaled test data

#     print("\n--- Classification Evaluation ---")

#     exact_accuracy = accuracy_score(y_test, y_pred)
#     print(f"Exact Match Accuracy: {exact_accuracy:.4f}")
#     f1_micro = f1_score(y_test, y_pred, average='micro', zero_division=0)
#     f1_macro = f1_score(y_test, y_pred, average='macro', zero_division=0)
#     f1_weighted = f1_score(y_test, y_pred, average='weighted', zero_division=0)
#     precision_micro = precision_score(y_test, y_pred, average='micro', zero_division=0)
#     recall_micro = recall_score(y_test, y_pred, average='micro', zero_division=0)
#     print(f"F1-Score (Micro): {f1_micro:.4f}")
#     print(f"F1-Score (Macro): {f1_macro:.4f}")
#     print(f"F1-Score (Weighted): {f1_weighted:.4f}")
#     print(f"Precision (Micro): {precision_micro:.4f}")
#     print(f"Recall (Micro): {recall_micro:.4f}")
#     print("\nDetailed Classification Report (per label):")
#     print(classification_report(y_test, y_pred, target_names=class_names, zero_division=0))

#     # --- Step 5: Display Test Examples ---
#     image_base_paths_for_display = {
#         'train': os.path.join('train', 'images'),
#         'valid': os.path.join('valid', 'images'),
#         'test': os.path.join('test', 'images')
#     }
#     display_test_examples(pose_model, test_filenames, y_test, y_pred, class_names, image_base_paths_for_display, num_examples=5)

#     print("--- Squat Form Classification Finished ---")

# if __name__ == "__main__":
#     # This will now always train with the tuned Balanced Random Forest
#     classify_squat_form()

Output Analyzed Video

## Image Analyzer

In [ ]:
image = '/content/Squat_image.jpg' # input image path to analyze

LOAD_CLASSIFIER_TYPE = "balanced_rfc"
TRAINED_CLASSIFIER_SAVE_PATH = "models/squat_classifier_balanced_rfc.joblib"
TRAINED_CLASS_NAMES_PATH = "models/squat_classifier_balanced_rfc_class_names.joblib"
TRAINED_SCALER_SAVE_PATH = "models/squat_classifier_scaler.joblib"

# --- Main Image Analysis Function ---
def analyze_image_squat_form(image_path, output_image_name="output_analyzed_image.jpg"):
    print(f"--- Starting Image Analysis for: {image_path} ---")

    # Load models
    try:
        pose_model = YOLO(model_checkpoint_path)
        print(f"Successfully loaded pose estimation model from: {model_checkpoint_path}")
    except Exception as e:
        print(f"ERROR: Could not load pose estimation model. Please check POSE_MODEL_PATH: {e}")
        return

    try:
        classifier = joblib.load(TRAINED_CLASSIFIER_SAVE_PATH)
        class_names = joblib.load(TRAINED_CLASS_NAMES_PATH)
        scaler = joblib.load(TRAINED_SCALER_SAVE_PATH)
        print(f"Successfully loaded classifier ({LOAD_CLASSIFIER_TYPE}), class names, and scaler.")
        print(f"Loaded class names: {class_names}")
    except Exception as e:
        print(f"ERROR: Could not load classifier, class names, or scaler. Please check paths: {e}")
        return

    # Read the image
    img_bgr = cv2.imread(image_path)
    if img_bgr is None:
        print(f"ERROR: Could not read image file: {image_path}. Please check path.")
        return

    img_height, img_width, _ = img_bgr.shape
    img_copy = img_bgr.copy() # Create a copy to draw on

    # 1. Pose Estimation
    results = pose_model(img_copy, conf=CONF_THRESHOLD, verbose=False)
    keypoints_xyc_to_draw = None
    features_for_classifier = None
    predicted_tags = []

    if results and len(results[0].boxes) > 0:
        best_box_idx = -1
        max_box_conf = -1.0
        # Find the person with the highest detection confidence
        for b_idx, box in enumerate(results[0].boxes):
            if box.conf.item() > max_box_conf:
                max_box_conf = box.conf.item()
                best_box_idx = b_idx

        if best_box_idx != -1 and results[0].keypoints is not None and len(results[0].keypoints.data) > best_box_idx:
            keypoints_xyc_data = results[0].keypoints.data[best_box_idx].cpu().numpy()

            # 2. Extract Features
            features_for_classifier = extract_keypoint_features(keypoints_xyc_data, img_width, img_height)

            # 3. Classify (only if features are valid length)
            if features_for_classifier is not None and len(features_for_classifier) == NUM_FEATURES_EXPECTED:
                try:
                    # Reshape for single prediction and APPLY SCALER
                    features_scaled = scaler.transform(features_for_classifier.reshape(1, -1))
                    predictions = classifier.predict(features_scaled)[0]
                    # Get active tags
                    predicted_tags = [class_names[i] for i, pred in enumerate(predictions) if pred == 1]
                except Exception as e:
                    print(f"Error during classification: {e}")
                    predicted_tags = ["CLASSIF. ERROR"]
            else:
                print("WARNING: No valid pose features extracted for classification.")

            # 4. Draw Keypoints and Skeleton (using CONF_THRESHOLD for drawing)
            SKELETON_CONNECTIONS = [
                (0, 1), (0, 2), (1, 3), (2, 4), (3, 5), (4, 6), (5, 7), (7, 9),
                (6, 8), (8, 10), (5, 11), (6, 12), (11, 13), (13, 15), (12, 14),
                (14, 16), (15, 17), (16, 18), (11, 12)
            ]
            KEYPOINT_COLOR = (0, 255, 0) # Green
            LINE_COLOR = (255, 0, 0) # Red
            RADIUS = 5
            THICKNESS = 2

            for kpt_idx, (x, y, conf) in enumerate(keypoints_xyc_data):
                if conf > CONF_THRESHOLD:
                    cv2.circle(img_copy, (int(x), int(y)), RADIUS, KEYPOINT_COLOR, -1)

            for connection in SKELETON_CONNECTIONS:
                p1_idx, p2_idx = connection
                p1_coords = get_kpt_coords(p1_idx, keypoints_xyc_data, CONF_THRESHOLD)
                p2_coords = get_kpt_coords(p2_idx, keypoints_xyc_data, CONF_THRESHOLD)
                if p1_coords is not None and p2_coords is not None:
                    cv2.line(img_copy, (int(p1_coords[0]), int(p1_coords[1])),
                             (int(p2_coords[0]), int(p2_coords[1])), LINE_COLOR, THICKNESS)
    else:
        print("No person detected in the image.")

    # 5. Overlay Predicted Tags
    TEXT_COLOR = (255, 255, 255) # White
    TEXT_BACKGROUND_COLOR = (0, 0, 0) # Black
    FONT = cv2.FONT_HERSHEY_SIMPLEX
    FONT_SCALE = 0.7
    FONT_THICKNESS = 2
    text_line_height = 30
    y_offset = 30

    if predicted_tags:
        predicted_tags_sorted = sorted(predicted_tags)
        for j, tag in enumerate(predicted_tags_sorted):
            text_to_display = f"Pred: {tag}"
            (text_width, text_height), baseline = cv2.getTextSize(text_to_display, FONT, FONT_SCALE, FONT_THICKNESS)
            cv2.rectangle(img_copy, (10, y_offset + j * text_line_height),
                          (10 + text_width, y_offset + j * text_line_height - text_height - baseline),
                          TEXT_BACKGROUND_COLOR, -1)
            cv2.putText(img_copy, text_to_display, (10, y_offset + j * text_line_height),
                        FONT, FONT_SCALE, TEXT_COLOR, FONT_THICKNESS, cv2.LINE_AA)
    else:
        text_to_display = "Pred: No tags detected"
        (text_width, text_height), baseline = cv2.getTextSize(text_to_display, FONT, FONT_SCALE, FONT_THICKNESS)
        cv2.rectangle(img_copy, (10, 30), (10 + text_width, 30 - text_height - baseline), TEXT_BACKGROUND_COLOR, -1)
        cv2.putText(img_copy, text_to_display, (10, 30),
                    FONT, FONT_SCALE, TEXT_COLOR, FONT_THICKNESS, cv2.LINE_AA)

    # Save the output image
    output_dir = os.path.dirname(output_image_name)
    if output_dir and not os.path.exists(output_dir):
        os.makedirs(output_dir)

    cv2.imwrite(output_image_name, img_copy)
    print(f"Analyzed image saved to: {os.path.abspath(output_image_name)}")

    # Display the image using Matplotlib (optional, for interactive environments)
    img_rgb = cv2.cvtColor(img_copy, cv2.COLOR_BGR2RGB)
    plt.figure(figsize=(10, 8))
    plt.imshow(img_rgb)
    plt.title(f"Analyzed Squat Form: {os.path.basename(image_path)}")
    plt.axis('off')
    plt.show()

    print("--- Image Analysis Complete ---")


if __name__ == "__main__":
    # Example Usage:
    # Ensure you have a sample image in a known path.
    # For demonstration, I'll use a placeholder. Replace with your actual image.
    sample_input_image = image
    output_image_file = "runs/analyzed_images/analyzed_sample_squat.jpg"

    # Ensure the output directory exists
    if not os.path.exists(os.path.dirname(output_image_file)):
        os.makedirs(os.path.dirname(output_image_file))

    analyze_image_squat_form(sample_input_image, output_image_file)


## Video Analyzer (Embeded)

In [ ]:
Video = '/content/Squat_valgus_demo.mp4' # path to the video file to analyze
# Path to save the trained classifier
TRAINED_CLASSIFIER_SAVE_PATH = "models/squat_classifier_balanced_rfc.joblib"
TRAINED_CLASS_NAMES_PATH = "models/squat_classifier_balanced_rfc_class_names.joblib"
TRAINED_SCALER_SAVE_PATH = "models/squat_classifier_scaler.joblib"

def analyze_video_stream(video_path, output_video_name="output_analyzed_video.mp4"):
    print(f"--- Starting Video Analysis for: {video_path} ---")

    # Load models
    try:
        pose_model = YOLO(model_checkpoint_path)
        print(f"Successfully loaded pose estimation model from: {model_checkpoint_path}")
    except Exception as e:
        print(f"ERROR: Could not load pose estimation model. Please check model_checkpoint_path: {e}")
        return

    try:
        classifier = joblib.load(TRAINED_CLASSIFIER_SAVE_PATH)
        class_names = joblib.load(TRAINED_CLASS_NAMES_PATH)
        # Load the scaler
        scaler = joblib.load(TRAINED_SCALER_SAVE_PATH) # Added scaler loading
        print(f"Successfully loaded classifier, class names, and scaler.")
        print(f"Loaded class names: {class_names}")
    except Exception as e:
        print(f"ERROR: Could not load classifier, class names, or scaler. Please check paths: {e}")
        return

    # Open video file
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"ERROR: Could not open video file: {video_path}. Please check path.")
        return

    # Get video properties
    frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    fourcc = cv2.VideoWriter_fourcc(*'mp4v') # Codec for .mp4

    # Setup VideoWriter
    # Ensure the output directory exists based on output_video_name
    output_dir = os.path.dirname(output_video_name)
    if output_dir and not os.path.exists(output_dir): # Only create if a directory is specified and doesn't exist
        os.makedirs(output_dir)
    output_video_path = output_video_name # The final path for the output video
    out = cv2.VideoWriter(output_video_path, fourcc, fps, (frame_width, frame_height))
    if not out.isOpened():
        print(f"ERROR: Could not create video writer for output: {output_video_path}. Check codec or path permissions.")
        cap.release()
        return

    SKELETON_CONNECTIONS = [
        (0, 1), (0, 2), (1, 3), (2, 4), (3, 5), (4, 6), (5, 7), (7, 9),
        (6, 8), (8, 10), (5, 11), (6, 12), (11, 13), (13, 15), (12, 14),
        (14, 16), (15, 17), (16, 18), (11, 12)
    ]
    KEYPOINT_COLOR = (0, 255, 0) # Green for keypoints
    LINE_COLOR = (255, 0, 0) # Red for skeleton
    RADIUS = 5
    THICKNESS = 2
    TEXT_COLOR = (255, 255, 255) # White
    TEXT_BACKGROUND_COLOR = (0, 0, 0) # Black
    FONT = cv2.FONT_HERSHEY_SIMPLEX
    FONT_SCALE = 0.7
    FONT_THICKNESS = 2

    frame_count = 0
    start_time = time.time()
    frames_to_display = [] # To store frames for matplotlib animation in Colab

    print("\nProcessing video frames...")
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        frame_count += 1
        img_copy = frame.copy() # Create a copy to draw on

        # 1. Pose Estimation
        results = pose_model(img_copy, conf=CONF_THRESHOLD, verbose=False)
        keypoints_xyc_to_draw = None
        features_for_classifier = None
        predicted_tags = []

        if results and len(results[0].boxes) > 0:
            best_box_idx = -1
            max_box_conf = -1.0
            # Find the person with the highest detection confidence
            for b_idx, box in enumerate(results[0].boxes):
                if box.conf.item() > max_box_conf:
                    max_box_conf = box.conf.item()
                    best_box_idx = b_idx

            if best_box_idx != -1 and results[0].keypoints is not None and len(results[0].keypoints.data) > best_box_idx:
                keypoints_xyc_data = results[0].keypoints.data[best_box_idx].cpu().numpy()

                # 2. Extract Features
                features_for_classifier = extract_keypoint_features(keypoints_xyc_data, frame_width, frame_height)

                # 3. Classify (only if features are valid length)
                if features_for_classifier is not None and len(features_for_classifier) == NUM_FEATURES_EXPECTED:
                    try:
                        # Reshape for single prediction and APPLY SCALER
                        features_scaled = scaler.transform(features_for_classifier.reshape(1, -1)) # Scaled features
                        predictions = classifier.predict(features_scaled)[0] # Predict with scaled features
                        # Get active tags
                        predicted_tags = [class_names[i] for i, pred in enumerate(predictions) if pred == 1]
                    except Exception as e:
                        print(f"Error during classification: {e}")
                        predicted_tags = ["CLASSIF. ERROR"]
                else:
                    pass # Keep silent for smooth video processing

                # 4. Draw Keypoints and Skeleton (using CONF_THRESHOLD for drawing)
                for kpt_idx, (x, y, conf) in enumerate(keypoints_xyc_data):
                    if conf > CONF_THRESHOLD: # Use CONF_THRESHOLD for drawing visibility
                        cv2.circle(img_copy, (int(x), int(y)), RADIUS, KEYPOINT_COLOR, -1)

                for connection in SKELETON_CONNECTIONS:
                    p1_idx, p2_idx = connection
                    p1_coords = get_kpt_coords(p1_idx, keypoints_xyc_data, CONF_THRESHOLD)
                    p2_coords = get_kpt_coords(p2_idx, keypoints_xyc_data, CONF_THRESHOLD)
                    if p1_coords is not None and p2_coords is not None:
                        cv2.line(img_copy, (int(p1_coords[0]), int(p1_coords[1])),
                                 (int(p2_coords[0]), int(p2_coords[1])), LINE_COLOR, THICKNESS)

        # 5. Overlay Predicted Tags
        if predicted_tags:
            text_line_height = 30
            y_offset = 30
            predicted_tags_sorted = sorted(predicted_tags)
            for j, tag in enumerate(predicted_tags_sorted):
                text_to_display = f"Pred: {tag}"
                (text_width, text_height), baseline = cv2.getTextSize(text_to_display, FONT, FONT_SCALE, FONT_THICKNESS)
                cv2.rectangle(img_copy, (10, y_offset + j * text_line_height),
                              (10 + text_width, y_offset + j * text_line_height - text_height - baseline),
                              TEXT_BACKGROUND_COLOR, -1)
                cv2.putText(img_copy, text_to_display, (10, y_offset + j * text_line_height),
                            FONT, FONT_SCALE, TEXT_COLOR, FONT_THICKNESS, cv2.LINE_AA)
        else:
            text_to_display = "Pred: No tags detected"
            (text_width, text_height), baseline = cv2.getTextSize(text_to_display, FONT, FONT_SCALE, FONT_THICKNESS)
            cv2.rectangle(img_copy, (10, 30), (10 + text_width, 30 - text_height - baseline), TEXT_BACKGROUND_COLOR, -1)
            cv2.putText(img_copy, text_to_display, (10, 30),
                        FONT, FONT_SCALE, TEXT_COLOR, FONT_THICKNESS, cv2.LINE_AA)

        img_rgb = cv2.cvtColor(img_copy, cv2.COLOR_BGR2RGB)
        frames_to_display.append(img_rgb)

        out.write(img_copy)

    cap.release()
    out.release()
    end_time = time.time()
    print(f"\nVideo processing complete. Processed {frame_count} frames in {end_time - start_time:.2f} seconds.")
    print(f"Output video saved to: {os.path.abspath(output_video_path)}")

    fig, ax = plt.subplots(figsize=(frame_width / 100, frame_height / 100))
    ax.axis('off')
    im = ax.imshow(frames_to_display[0])

    def animate(i):
        im.set_array(frames_to_display[i])
        return [im]

    ani = animation.FuncAnimation(fig, animate, frames=len(frames_to_display), interval=1000/fps, blit=True)
    plt.close(fig)

    display(HTML(ani.to_html5_video()))
    print("Video display complete.")

if __name__ == "__main__":
    input_video_path = Video
    # This is the path where the MP4 file will be saved.
    output_video_file = "runs/analyzed_videos/analyzed_squat_clip.mp4"

    if not os.path.exists('models'):
        os.makedirs('models')

    if not os.path.exists(os.path.dirname(output_video_file)):
        os.makedirs(os.path.dirname(output_video_file))

    analyze_video_stream(input_video_path, output_video_file)
